# **MÓDULO 1: DATA LAKE Y SU MANTENIMIENTO**
## IONClinics & Universidad Complutense de Madrid

## **Pre-Work Checklist: Ejecutar antes de comenzar**

Esta etapa inicial prepara el entorno de trabajo para la ejecución del sistema. Aquí se cargan las librerías necesarias, se configuran las rutas a las bases de datos y archivos requeridos, y se establecen las sesiones de Spark. Este paso es fundamental para garantizar que todos los procesos posteriores se ejecuten de manera fluida y sin errores relacionados con la configuración o dependencias del entorno.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install semanticscholar
!pip install pymupdf
!pip install requests
!pip install pdfplumber

In [ ]:
from configparser import ConfigParser
from pathlib import Path
import requests
import urllib
import json
import pandas as pd
from semanticscholar import SemanticScholar
import os
import pdfplumber
import fitz
from datetime import datetime

In [ ]:
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
#Check this site for the latest download link https://www.apache.org/dyn/closer.lua/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!wget -q https://dlcdn.apache.org/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz
!pip install -q findspark
!pip install --upgrade pyspark
!pip install py4j
!pip install --upgrade openai

import sys
import pyspark.sql.types as T
import pyspark.sql.functions as F
from pyspark.sql.functions import col, explode, from_json, substring, split, udf,lower, expr, regexp_replace, monotonically_increasing_id, trim
from pyspark.sql.types import StringType, StructType, StructField, IntegerType, ArrayType
from pyspark.sql.utils import AnalysisException
from pyspark.sql import Row

import openai
from openai import OpenAI
import re
import ast

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [2,901 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,722 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [8,927 kB]
Get:14 http://

### **Parameter setting**

PATHS

In [ ]:
# Path for storage
dtset_dir = Path('/content/drive/MyDrive/Data Lake IONClinics/S2ORC completo')
dir_data = Path('/content/drive/MyDrive/Data Lake IONClinics/S2ORC completo/20240917/rawdata')
dtset_dir_parquet = Path('/content/drive/MyDrive/Data Lake IONClinics/S2ORC completo/spark')

In [ ]:
# PROMPT Context Databases
path_df_BBDD_nonprocessed_example_csv = Path('/content/drive/MyDrive/Data Lake IONClinics/Actualizaciones/Contexto GPT NER/df_BBDD_nonprocessed_example')
path_df_BBDD_example = Path('/content/drive/MyDrive/Data Lake IONClinics/Actualizaciones/Contexto GPT NER/df_BBDD_example')

# Periodic Updates Reportings
path_historial = Path('/content/drive/MyDrive/Data Lake IONClinics/Actualizaciones/Historial.xlsx')
path_updates = Path('/content/drive/MyDrive/Data Lake IONClinics/Actualizaciones/Tabla_de_actualizaciones.xlsx')

# Database to be updated
path_BBDD = Path('/content/drive/MyDrive/Data Lake IONClinics/Actualizaciones/BBDD_actualizado_urls_mayo_2025.xlsx')

# Updates from new releases Download folder
dtset_updates_dir_or = '/content/drive/MyDrive/Data Lake IONClinics/Actualizaciones/updates'

# Urls record list
path_urls_record = Path('/content/drive/MyDrive/Data Lake IONClinics/Actualizaciones/updates/lista_urls.xlsx')

# RPA papers Excel
path_rpa_papers = Path('/content/drive/MyDrive/Data Lake IONClinics/Actualizaciones/updates/df_tdcs_vacios.xlsx')

API KEYS

In [ ]:
# INSERT HERE YOUR CREDENTIALS
GPT_api_key =   ## Introducir aquí llave API de OpenAI
Semantic_Scholar_api_key =  ## Introducir aquí llave API de Semantic Scholar

SYMPTOMS KEYWORD LIST

In [ ]:
keyword_list_symptoms = ["stroke", "pain", "fibromyalgia", "depression","schizophrenia"]

### **Spark Session**

In [ ]:
from pyspark.sql import SparkSession
import findspark

# Initialize Spark
findspark.init()

# Create or get the SparkSession with custom configurations
spark = SparkSession.builder \
    .appName("tDCS_recomendador") \
    .config("spark.executor.instances", "4") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# Create an RDD with increased partitions
rdd = spark.sparkContext.parallelize(range(100), numSlices=8)  # Increase the number of partitions

# Get the number of partitions
num_partitions = rdd.getNumPartitions()
print("Number of partitions (indirect indicator of workers):", num_partitions)


Number of partitions (indirect indicator of workers): 8


In [ ]:
spark

In [ ]:
dtset_dir_parquet = Path(dtset_dir_parquet)
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
hdfs_dir_parquet = spark._jvm.org.apache.hadoop.fs.Path(dtset_dir_parquet.as_posix())
if not fs.exists(hdfs_dir_parquet):
    fs.mkdirs(hdfs_dir_parquet)

## **PROCESO 1: Descarga y Procesamiento Inicial**

### **Obtención del corpus desde Semantic Scholar**

Este proceso consiste en adquirir y almacenar la base de datos de textos científicos proveniente de Semantic Scholar. Las etapas principales son:

1. Descargar el corpus completo de textos proporcionado por Semantic Scholar.

2. Guardar el corpus en la nube para asegurar su disponibilidad y facilitar el acceso durante los procesos posteriores de análisis y extracción de información.


In [ ]:
def download_S2(token, dest_dir, version="latest"):
    """
    Download SemanticScholar info into `dest_dir`/`version`
    """
    dest_dir = Path(dest_dir)
    # Download information about latest version
    data = requests.get('https://api.semanticscholar.org/datasets/v1/release/' + version + '/')

    if data.ok:

        jsonData = data.json()

        release = jsonData['release_id']
        datasets = [d['name'] for d in jsonData['datasets']]

        dest_dir = dest_dir.joinpath(release.replace('-','')).joinpath('rawdata')
        dest_dir.mkdir(parents=True, exist_ok=True)

        with dest_dir.joinpath('release_info.json').open('w') as jsonFile:
            json.dump(jsonData, jsonFile, indent=4)

        for dtset in ['s2orc']:
            url = "http://api.semanticscholar.org/datasets/v1/release/" + release + "/dataset/"+dtset
            dtset_info = requests.get(url, headers={'x-api-key':token})

            if dtset_info.ok:
                print("\nDownloading dataset " + dtset)
                dtsetData = dtset_info.json()

                dtset_dir = dest_dir.joinpath(dtset)
                dtset_dir.mkdir(parents=False, exist_ok=True)

                with dtset_dir.joinpath(dtset+'info.json').open('w') as jsonFile:
                    json.dump(dtsetData, jsonFile, indent=4)

                for idx, fileUrl in enumerate(dtsetData['files'][62:92]):  ## creo que aqui se pone el número para ca
                    print("Downloading file", idx, "of", len(dtsetData['files']))
                    fileName = dtset_dir.joinpath(dtset + "-part" + str(idx) + ".json.gz")
                    if not fileName.is_file():
                        urllib.request.urlretrieve(fileUrl, fileName)

    return


if __name__ == "__main__":
    download_S2(token = Semantic_Scholar_api_key, dest_dir = dtset_dir)

In [ ]:
download_S2(Semantic_Scholar_api_key, dir_data, version="latest")

### **Extracción de documentos relacionados con tDCS**

En este proceso se depura y filtra el corpus inicial para identificar únicamente los textos relevantes para el estudio de la estimulación transcraneal por corriente directa (tDCS). Las acciones principales incluyen:

1. **Eliminar las secciones de agradecimientos y referencias** del texto, con el fin de conservar únicamente el contenido científico relevante.

2. **Filtrar los artículos que contienen la palabra clave 'tDCS'**, asegurando que solo se analicen documentos directamente relacionados con esta técnica.

In [ ]:
def filter_kw(dataframe, text_column, keyword_list):  ## Filter function intended just for the 'tDCS' word
    def count_kwds(text):
        if text is None:
            return 0
        else:
            return sum(text.lower().count(k) for k in keyword_list)

    count_kwds_udf = udf(count_kwds, IntegerType())
    dataset = dataframe.withColumn("Kwd_count", count_kwds_udf(dataframe[text_column]))
    dataset = dataset.filter(dataset.Kwd_count > 25).cache()
    return dataset

In [ ]:
def filter_acknowledgements(df, text_column):  ## Remove acknowledgments and reference section from text
    df = df.withColumn(text_column, regexp_replace(df[text_column], '\n', ' '))
    df_filtered = df.withColumn(
        text_column,
        expr("CASE \
             WHEN INSTR(lower({}), 'acknowledgments') > 0 THEN substring_index(lower({}), 'acknowledgments', 1) \
             WHEN INSTR(lower({}), 'conflicts of interest') > 0 THEN substring_index(lower({}), 'conflicts of interest', 1) \
             WHEN INSTR(lower({}), '[crossref]') > 0 THEN substring_index(lower({}), '[crossref]', 1) \
             WHEN INSTR(lower({}), '10.') > 0 THEN substring_index(lower({}), '10.', 3) \
             ELSE {} END".format(text_column, text_column, text_column, text_column, text_column,text_column, text_column, text_column, text_column ))
    )
    return df_filtered


In [ ]:
def extract_text(dir_papers):  ##  def extract_text(dir_papers, x) -- Use this if you want to filter by batches
    df_text_abstract_filter_tdcs_concat = None

    try:
        # Get a list of the JSON files in the directory
        json_files = sorted(dir_papers.glob('*.json.gz')) #[:x]  -- Use this if you want to filter by batches

        for json_file in json_files:
            try:
                df = spark.read.json('file:///' + json_file.as_posix())

                # Select relevant columns
                df_text = df.select('corpusid', col('content.text').alias('text'))

                # Filter by acknowledgements and keyword threshold
                keywords = ['tdcs']
                df_text_acknow = filter_acknowledgements(df_text, 'text') # Filter by acknowledgements
                df_text_abstract_filter_tdcs = filter_kw(df_text_acknow, 'text', keywords) # Filter by keyword

                # Concatenate with the previous filtered dataframe
                if df_text_abstract_filter_tdcs_concat is not None:
                    df_text_abstract_filter_tdcs_concat = df_text_abstract_filter_tdcs_concat.union(df_text_abstract_filter_tdcs)
                else:
                    df_text_abstract_filter_tdcs_concat = df_text_abstract_filter_tdcs

            except AnalysisException as e:
                print(f'Error reading data from {json_file.name}: {e}')

    except Exception as e:
        print(f'Unexpected error: {e}')

    return df_text_abstract_filter_tdcs_concat

In [ ]:
## Colocar path de acuerdo al release id especifico de cuando hizo la descarga
## Por ejemplo --> dir_data = Path('/content/drive/MyDrive/Data Lake IONClinics/S2ORC completo/20240917/rawdata')

dir_data = Path('')
dir_papers = dir_data.joinpath('s2orc')
df_papers_tdcs = extract_text(dir_papers)

if df_papers_tdcs:
    print('Number of documents in dataset:', df_papers_tdcs.count())
    df_papers_tdcs.show(n=15, truncate=300, vertical=True)
else:
    print("No documents found after filtering.")

Unexpected error: An error occurred while calling o1470.json.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 90.0 failed 1 times, most recent failure: Lost task 0.0 in stage 90.0 (TID 104) (bc5715fbe2d2 executor driver): java.io.EOFException: Unexpected end of input stream
	at org.apache.hadoop.io.compress.DecompressorStream.decompress(DecompressorStream.java:165)
	at org.apache.hadoop.io.compress.DecompressorStream.read(DecompressorStream.java:105)
	at java.base/java.io.InputStream.read(InputStream.java:205)
	at org.apache.hadoop.util.LineReader.fillBuffer(LineReader.java:191)
	at org.apache.hadoop.util.LineReader.readDefaultLine(LineReader.java:227)
	at org.apache.hadoop.util.LineReader.readLine(LineReader.java:185)
	at org.apache.hadoop.mapreduce.lib.input.LineRecordReader.nextKeyValue(LineRecordReader.java:200)
	at org.apache.spark.sql.execution.datasources.RecordReaderIterator.hasNext(RecordReaderIterator.scala:39)
	at org.apache.spark.sql.exe

### **Filtrado por síntomas**

En esta etapa se refina aún más el conjunto de documentos seleccionando únicamente aquellos que están relacionados con síntomas o patologías de interés. El proceso consiste en:

Filtrar los textos mediante palabras clave asociadas a síntomas específicos, tales como:

* **Stroke**
* **Pain**
* **Fibromialgia**
* **Depresión**
* **Esquizofrenia**

Este filtrado permite enfocar el análisis en publicaciones científicas relevantes para las condiciones clínicas más comunes tratadas con tDCS.


In [ ]:
## Keyword filter

from pyspark.sql.functions import udf, col, substring, regexp_replace
from pyspark.sql.types import IntegerType

def filter_kwd(dataframe, text_column, symptom_keywords, end_index, threshold):
    # Define a user-defined function (UDF) to count keyword occurrences
    count_kwds_udf = udf(lambda text: sum(text.lower().count(keyword.lower()) for keyword in symptom_keywords), IntegerType())

    # Replace punctuation with spaces
    dataframe = dataframe.withColumn("cleaned_text", regexp_replace(substring(col(text_column), 1, end_index), "-", " "))

    #dataframe.show(n=15, truncate=300, vertical=True)

    # Add a column to the DataFrame containing the keyword counts
    dataframe = dataframe.withColumn("Kwd_count", count_kwds_udf(col("cleaned_text")))

    # Filter rows based on the keyword count threshold
    filtered_dataframe = dataframe.filter(col("Kwd_count") >= threshold)

    # Drop the temporary cleaned_text column if not needed
    filtered_dataframe = filtered_dataframe.drop("cleaned_text")

    return filtered_dataframe

In [ ]:
from pyspark.sql.functions import lit

def filter_symptoms(dataframe, text_column, symptom_keywords, end_index, threshold):

    filtered_dataframe = filter_kwd(dataframe, text_column, symptom_keywords, end_index, threshold)
    updated_rows = []

    #symptom_caused = {}
    symptom_list = []
    counter_list = []
    for row in filtered_dataframe.collect():
      if row[text_column] is not None:
          #print('-------------')
          #print('NEW ROW')
          corpusid = row['corpusid']
          text = row[text_column][:end_index]
          symptom_caused = {}
          max_symptom = 0
          max_counter = 0
          for keyword in symptom_keywords:
              counter = text.lower().count(keyword.lower())
              if counter >= (threshold):
                  if counter > max_counter:
                      max_counter = counter
                      max_symptom = keyword
          updated_rows.append((corpusid, text, max_symptom, max_counter))
      else:
         continue

    # Definir el esquema del DataFrame
    schema = StructType([
        StructField("corpusid", StringType(), True),
        StructField("text", StringType(), True),
        StructField("Symptom", StringType(), True),
        StructField("counter", IntegerType(), True)
    ])

    # Crear el DataFrame con el esquema definido
    df_updated = spark.createDataFrame(updated_rows, schema)
    df_updated = df_updated.filter(col("counter") > 1)

    return df_updated

In [ ]:
df_paper_tdcs_symptoms = filter_symptoms(df_papers_tdcs, 'text', keyword_list_symptoms,end_index= 4000, threshold=6)
print('Number of documents in dataset:', df_paper_tdcs_symptoms.count())
df_paper_tdcs_symptoms.show(n=15, truncate=300, vertical=True)

Number of documents in dataset: 284
-RECORD 0----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 corpusid | 221755084                                                                                                                                                                                                                                                                                                    
 text     |  effect of short period simultaneous stimulation of transcranial direct current stimulation (tdcs) on occupational therapy to motor function of upper extremity in stroke subjects   adrian utomo  departement of physical medicine and rehabilitation faculty of medicine department of physical medicin... 
 Symptom  | stroke    

In [ ]:
# Saving in the Data Lake
output_parquet_path_2 = dtset_dir_parquet.joinpath("df_filtrado_tdcs_symptoms")
df_paper_tdcs_symptoms.write.parquet(output_parquet_path_2.as_posix())
print(f"DataFrame saved as Parquet file at: {output_parquet_path_2}")

DataFrame saved as Parquet file at: /content/drive/MyDrive/Data Lake IONClinics/S2ORC completo/spark/df_filtrado_tdcs_symptoms_prueba


### **Función de reconocimiento de entidades con GPT**

Este proceso utiliza modelos de lenguaje (GPT) para identificar y extraer automáticamente entidades relevantes dentro de los textos científicos. Las etapas que lo componen son:

1. **Diseño del prompt:** se formula cuidadosamente una instrucción que guía al modelo para extraer información específica sobre protocolos de tDCS.

2. **Extracción en formato JSON:** el modelo responde estructurando los datos en formato JSON, lo cual facilita su interpretación y procesamiento automatizado.

3. **Generación de DataFrame:** los datos extraídos se organizan en un DataFrame para su análisis posterior e integración en el sistema de recomendación.

In [ ]:
from openai import OpenAI

def NER(text):
    client = OpenAI(api_key = GPT_api_key)
    response = client.chat.completions.create(
        model="gpt-4-turbo",
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": "As a natural language processing expert who responds using JSON, your task is to extract named entities related to tDCS interventions, including: 1- Current intensity in milliamps (mA) of the tDCS sessions (e.g. 2mA, 1mA). 2- Duration in minutes (min) of the tDCS sessions (e.g. 20 minutes, 30 minutes). 3- Session details such as any information about the periodicity of the intervention,  number of sessions (e.g. 10 sessions, 20 sessions), times per day, days per week and weeks or months. 4- Electrode placement in 10-20 system locations (e.g., C4, F3). 5- If the electrode placement is Ipsilateral o Contralateral to the affected region . 6- Stimulation modality (e.g., anodal tDCS, bilateral tDCS, high-definition tDCS). 7- Pathology being treated (e.g. aphasia, neuropathic pain). 8- Study/trial methodology design used (e.g., double-blinded, sham-controlled, RCT,review, meta-analysis). 9- A (YES/NO) whether the tDCS intervention showed experimental evidence of effectiveness or not. 10- Age range of the sample used in the intervention (e.g. 18-70, greater than 18). 11- Gender distribution of the sample (number of males and number of females). 12- Number of people used as samples. 13- Country or origin of the sample (e.g. Italy, Korea, Spain). 14- The title of the research paper. 15- The doi of the research papaer. 16- The year that was published the research paper. If any information is not available, please use 'Not specified' or an empty list."
            },
            {
                "role": "user",
                "content": "Please provide a research paper excerpt containing information about a tDCS intervention."
            },
            {
                "role": "assistant",
                "content": "Your extracted entities should be formatted as follows: {'Year': ['year'],'Title': ['title'], 'DOI': ['doi'], 'Current': ['current_intensity'], 'Duration': ['duration'],'Periodicity_info': ['periodicity_info'],'Sessions': ['number_sessions'], 'Times per day': ['times_per_day'], 'Days per week': ['days_per_week'], 'Weeks': ['weeks_months'], 'Cathode': ['cathode_placement_10_20'], 'Cathode_location': ['cathode_ipsilateral_contralateral']  , 'Anode': ['anode_placement_10_20'],'Anode_location': ['anode_ipsilateral_contralateral']  ,'Modality': ['stimulation_modality'],'Pathology': ['pathology'],'Evidence': ['trial_design'], 'Effectiveness': ['effectiveness_yes_no'], 'Age': ['age'], 'Males': ['males']'Females': ['females'], 'N_sample': ['n_sample'],'Origin': ['origin']}. If any information is not available, please use 'Not specified' or an empty list."
            },
            {
                "role": "user",
                "content": text
            }
        ],
        temperature=0,
        top_p=1,
        frequency_penalty=0,
        presence_penalty=0
    )
    result = response.choices[0].message.content.strip(" \n")
    return result

In [ ]:
## EXTRACT INFORMATION FROM JSON FORMAT
def extract_info_from_json(json_str):
    try:
        data = json.loads(json_str)
        extracted_info = {
            "Year": data.get("Year", [None])[0],
            "Title": data.get("Title", [None])[0],
            "DOI": data.get("DOI", [None])[0],
            "Current": data.get("Current", [None])[0],
            "Duration": data.get("Duration", [None])[0],
            "Periodicity_info": data.get("Periodicity_info", [None])[0],
            "Sessions": data.get("Sessions", [None])[0],
            "Times_per_day": data.get("Times per day", [None])[0],
            "Days_per_week": data.get("Days per week", [None])[0],
            "Weeks": data.get("Weeks", [None])[0],
            "Cathode": data.get("Cathode", [None])[0],
            "Cathode_location": data.get("Cathode_location", [None])[0],
            "Anode": data.get("Anode", [None])[0],
            "Anode_location": data.get("Anode_location", [None])[0],
            "Modality": data.get("Modality", [None])[0],
            "Pathology": data.get("Pathology", [None])[0],
            "Evidence": data.get("Evidence", [None])[0],
            "Effectiveness": data.get("Effectiveness", [None])[0],
            "Age": data.get("Age", [None])[0] if data.get("Age") else None,
            "Males": data.get("Males", [None])[0] if data.get("Males") else None,
            "Females": data.get("Females", [None])[0] if data.get("Females") else None,
            "N_sample": data.get("N_sample", [None])[0] if data.get("N_sample") else None,
            "Origin": data.get("Origin", [None])[0] if data.get("Origin") else None
        }
        return extracted_info
    except Exception as e:
        print("Error:", e)
        return None



In [ ]:
def parameter_extraction(df):
    df_protocolo_parameters = []
    references_keyword = "Acknowledgments"

    df_size = df.count()
    df_selected = df.head(df_size)
    #df_selected = df.head(50)  ## We use this line if we just want to prove the prompt for the first 50 papers in the list

    for record in df_selected:
        text = str(record.text.replace('\n', ' '))
        corpusid = record.corpusid
        symptom = record.Symptom

        # Find the index of the references keyword and truncate the text
        references_index = text.find(references_keyword)
        if references_index != -1:
            text = text[:references_index]

        # Extract information using NER_2 and convert to DataFrame row
        result = NER(text)

        extracted_info = extract_info_from_json(result)
        extracted_info['corpusid'] = corpusid
        df_protocolo_parameters.append(extracted_info)

    # Convert list of dictionaries to DataFrame
    df_protocolo_parameters = pd.DataFrame(df_protocolo_parameters)

    return df_protocolo_parameters


In [ ]:
df_protocolo_parameters = parameter_extraction(df_paper_tdcs_symptoms)
df_protocolo_parameters

,Title,DOI,Current,Duration,Sessions,Times_per_day,Days_per_week,Weeks,Cathode,Anode,Modality,Symptom_specific,Evidence,Effectiveness,corpusid,Symptom_general
0,Bihemispheric-tDCS and Upper Limb Rehabilitati...,10.3389/fnhum.2016.00258,1.5 mA,20 minutes,9 sessions,1,3,3,M1 contralateral to the non-paretic limb,M1 contralateral to the paretic limb,Bihemispheric-tDCS,Chronic stroke,"Double-blinded, randomized controlled trial",YES,405274,stroke
1,Analgesic Effects of Transcranial Direct Curre...,10.5392/IJOC.2012.8.1.074,0.1 mA,20 min/time,2 times/day,2,5,6,neck,cerebral cortex,tDCS,central neuropathic pain,behavioral test,YES,15396623,pain
2,Long-term effects of transcranial direct-curre...,10.3389/fnhum.2014.00785,1.5 mA,20 min,10 consecutive sessions,1,5,2,contralateral supraorbital region,"left frontal (perilesional) region, over the c...",anodal tDCS (a-tDCS),chronic post-stroke aphasia,single-blind study,YES,16141515,stroke
3,Putative Physiological Mechanisms Underlying t...,10.1097/ajp.0b013e318157233b,1 or 2 milliamperes,Several minutes,5 or 10 consecutive days,1,Daily,1 week (for 5 sessions) or 2 weeks (for 10 ses...,S1,M1,Anodal and Cathodal tDCS,Pain,Experimental settings,YES,18610206,pain
4,Effects of Transcranial Direct Current Stimula...,10.3390/bs12020037,1 mA,20 min,1 session per week,1,1,3,FP2,F3,tDCS,Fibromyalgia,Double-blind,YES,246673261,pain
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
460,Brain Circuits Involved in the Development of ...,10.3389/fneur.2021.732034,1 mA,20 min,1,1,None,None,None,Left M1,tDCS,Chronic musculoskeletal pain,Experimental,YES,237406696,stroke
461,Modulated Effectiveness of Rehabilitation Moti...,10.3389/fneur.2023.1200741,1.5 mA,20 min,15,1,5,3,Right shoulder,F3 (left dorsolateral prefrontal cortex),Anodal tDCS,Stroke,"Randomized, single-blind, controlled clinical ...",YES,259168244,stroke
462,Anodal Transcranial Direct Current Stimulation...,10.1038/s41598-017-00185-w,1 mA,20 minutes,1,1,Not specified,Not specified,FP2,F3,Anodal tDCS,Fibromyalgia,"Randomized, crossover blind, clinical trial",YES,3291381,stroke
463,Transcranial Direct Current Stimulation in Str...,10.3389/fpain.2021.696547,2 mA,30 seconds ramp up and ramp down,10 sessions,1,5,2,supraorbital region contralateral to the anode,C3 or C4,active-anodal tDCS or sham tDCS,shoulder pain in stroke survivors,"double-blind, randomized, controlled trial",NO,235701129,pain


In [ ]:
## Saving in the Data Lake as df_BBDD_original (as csv and Excel)
output_parquet_path_3 = dtset_dir_parquet.joinpath("df_gpt_parameters")
df_protocolo_parameters.to_csv(output_parquet_path_3, index=False)
print(f"DataFrame saved as Parquet file at: {output_parquet_path_3}")

DataFrame saved as Parquet file at: /content/drive/MyDrive/Data Lake IONClinics/S2ORC completo/spark/df_gpt_parametros_prueba


### **Estandarización de parámetros del protocolo**

Dada la variabilidad en los formatos de respuesta generados por el modelo GPT, se implementó un proceso de estandarización para unificar y normalizar la salida. Este paso es fundamental para asegurar la coherencia y calidad de los datos antes de su análisis. Los parámetros que se estandarizan incluyen:

1. **Corriente aplicada (Current)**

2. **Duración de la sesión (Duration)**

3. **Frecuencia del tratamiento:** número de veces al día, días por semana, y semanas totales

4. **Modalidad de estimulación (Modality)**

5. **Ubicación de electrodos según el sistema 10/20**

6. **Patología tratada (Pathology)**

7. **Síntoma abordado (Symptom)**

8. **Nivel de evidencia científica (Scientific Evidence)**

9. **Efectividad del tratamiento (Treatment Effectiveness)**

10. **Información Sociodemográfica:** edad (clasificando en los diferentes grupos de edad), género y origen.

In [ ]:
def standardize_current(current):
    current = str(current).lower()

    keywords_2 = ["2.0", "1-2", "1 to 2", "0.5 to 2", "1 or 2", "1/2", "1 - 2", '1 to 2', '0.5-2', '2mA','2ma']
    keywords_1 = ["1.0",'1mA','1ma']
    keywords_1_5 = ["1.5",'1.5mA','1.5ma']
    keywords_none  = ["none", "not",'nan']

    if any(keyword in current for keyword in keywords_2):
        return 2.0
    elif any(keyword in current for keyword in keywords_1):
        return 1.0
    elif any(keyword in current for keyword in keywords_1):
        return 1.5
    elif any(keyword in current for keyword in keywords_none):
        return 0.0 #'Not specified'

    else:
        if 'µa' in current:
            return 0.0
            #try:
             # num = float(current.replace('µa', '').strip())
            #return 0.0  #'others' #num / 1000  # Convert µA to mA
        else:
            try:
                return float(current)
            except ValueError:
                return 0.0 #'others' #current

In [ ]:
def standarize_duration(duration):

    duration = str(duration).lower()


    keywords_20 = ["10 to 20", "15 to 20","10-20","15-20","up to 20",'1200 seconds','7-20','13 to 20','5 to 20']
    keywords_25 = ['25']
    keywords_30  = ["20 to 30", "25 to 30","20-30","25-30","15-30","5-30","up to 30",'half an hour','10-30','8 to 30','13 to 30','5 to 30','10 to 30']
    keywords_40  = ["30-40", "up to 40",'10-40','20-40']
    keywords_60  = ["50-60","30-60","up to 60", 'an hour']
    keywords_none  = ["none", "not",'nan']

    if any(keyword in duration for keyword in keywords_20):
        return 20.0

    elif any(keyword in duration for keyword in keywords_25):
        return 25.0

    elif any(keyword in duration for keyword in keywords_30):
        return 30.0

    elif any(keyword in duration for keyword in keywords_40):
        return 40.0

    elif any(keyword in duration for keyword in keywords_60):
        return 60.0

    elif any(keyword in duration for keyword in keywords_none):
        return 0.0 #'Not specified'

    else:
        try:
           return float(duration)
        except ValueError:
              return 0.0 #'others' #duration

In [ ]:
def standarize_times_perday(time):

    time = str(time).lower()


    keywords_2_times = ["2 times", "twice a day","twice","2 sessions",'2 times/day']
    keywords_1_time = ['once','1 session per day','1 session/day','1x per day','daily']
    keywords_hourly  = ["hourly", "every hour"]
    keywords_none  = ["none", "not",'nan']


    if any(keyword in time for keyword in keywords_2_times):
        return 2.0

    elif any(keyword in time for keyword in keywords_1_time):
        return 1.0

    elif any(keyword in time for keyword in keywords_hourly):
        return 24

    elif any(keyword in time for keyword in keywords_none):
        return  0.0 #'Not specified'

    else:
        try:
           return float(time)
        except ValueError:
              return 0.0 #'others'  #time #'not specified'

In [ ]:
def standarize_days_perweek(days):

    days = str(days).lower()


    keywords_5_days = ["5 consecutive","5 times", "monday to friday","5 days","five days","five-day","workday","daily","to 5",'5','weekdays','five times a week','to 5']
    keywords_2_days = ['twice weekly','2 times',"2 days","two","2 consecutive",'twice per week','to 2','-2']
    keywords_3_days  = ["3 times", "3 days in a week",'3 consecutive','3 days per week','3 days','-3','to 3']
    keywords_none  = ["none", "not",'nan']
    keywords_7_days  = ["7","including weekends","7 days",'-7','to 7']
    keywords_6_days  = ["6","6 days"]
    keywords_1_days  = ["once per week","once weekly"]


    if any(keyword in days for keyword in keywords_5_days):
        return 5.0

    elif any(keyword in days for keyword in keywords_2_days):
        return 2.0

    elif any(keyword in days for keyword in keywords_3_days):
        return 3.0

    elif any(keyword in days for keyword in keywords_none):
        return 0.0 #'Not specified'

    elif any(keyword in days for keyword in keywords_7_days):
        return 7.0

    elif any(keyword in days for keyword in keywords_6_days):
        return 6.0

    elif any(keyword in days for keyword in keywords_1_days):
        return 1.0

    else:
        try:
           return float(days)
        except ValueError:
              return 0.0 #days #'not specified'

In [ ]:
def standarize_weeks(week):

    week = str(week).lower()


    keywords_5_weeks = ["5"]
    keywords_2_weeks = ['1-2','1 to 2',"2"]
    keywords_3_weeks  = ["2 to 3", "2-3"]
    keywords_none  = ["none", "not",'nan']
    keywords_7_weeks  = ["7"]
    keywords_6_weeks  = ["6"]
    keywords_4_weeks  = ["4",'one month','']
    keywords_8_weeks  = ["8",'two months']
    keywords_12_weeks  = ["12"]


    if any(keyword in week for keyword in keywords_5_weeks):
        return 5.0

    elif any(keyword in week for keyword in keywords_2_weeks):
        return 2.0

    elif any(keyword in week for keyword in keywords_3_weeks):
        return 3.0

    elif any(keyword in week for keyword in keywords_none):
        return 0.0 #'Not specified'

    elif any(keyword in week for keyword in keywords_7_weeks):
        return 7.0

    elif any(keyword in week for keyword in keywords_6_weeks):
        return 6.0

    elif any(keyword in week for keyword in keywords_4_weeks):
        return 4.0

    elif any(keyword in week for keyword in keywords_8_weeks):
        return 8.0

    elif any(keyword in week for keyword in keywords_12_weeks):
        return 12.0

    else:
        try:
            return float(week)
        except ValueError:
              return 0.0 #week #'not specified'

In [ ]:
def standarize_modality(modality):

    modality = str(modality).lower()

    keywords_bilateral = ["bi-", "bihemisphere","bihemispheric","bilateral","dual","bi-tdcs","anodal and cathodal",'bimodal','bifrontal','bicephalic']
    keywords_anodal  = ["anodal", "a-tdcs","atdcs",'anodic','conventional','tdcs']
    keywords_cathodal  = ["cathodal", "c-tdcs",'ctdcs']
    keywords_multifocal  = ["multifocal",'multichannel']
    keywords_hd  = ["high-definition", "h-tdcs","htdcs","hd-tdcs"]
    keywords_tdcs_combined = ['tdcs combined','tdcs +', 'tdcs with','combined']
    keywords_cerebellar = ['cerebellar']
    keywords_none  = ["none", "not",'nan']
    keywords_tms  = ["tms",'magnetic stimulation']


    if any(keyword in modality for keyword in keywords_bilateral):
        return "Bilateral"

    elif any(keyword in modality for keyword in keywords_cathodal):
        return "Cathodal"

    elif any(keyword in modality for keyword in keywords_hd):
        return "High-definition"

    elif any(keyword in modality for keyword in keywords_multifocal):
        return "Multifocal"

    elif any(keyword in modality for keyword in keywords_tdcs_combined):
        return "tDCS combined"

    elif any(keyword in modality for keyword in keywords_cerebellar):
        return "Cerebellar"

    elif any(keyword in modality for keyword in keywords_tms):
        return "TMS"

    elif any(keyword in modality for keyword in keywords_anodal):
        return "Anodal"

    elif any(keyword in modality for keyword in keywords_none):
        return 'Not specified'

    else:
        return 'others' #modality

In [ ]:
def standarize_pathology(pathology):
    pathology = str(pathology).lower()

    keywords_fibromyalgia = ["fibromyalgia"]
    keywords_stroke = ["ischemic", "ischemic stroke", "acute ischemic stroke",'subacute stroke', 'sub-acute stroke','acute stroke','stroke','unilateral spatial neglect','visuospatial neglect','unilateral visuospatial neglect','hemispatial neglect','visuospatial neglect','spatial neglect','aphasia','hemiparesis','intracerebral hemorrhage', 'cerebral vasospasm','swallowing function']
    keywords_pain = ["pain",'hyperalgesia','temporomandibular','arthralgia','neuralgia','headache','migraine','trigeminal','restless legs syndrome']
    keywords_schizophrenia = ["schizophrenia",'catatonia']
    keywords_depression = ["depression", 'depressive']
    keywords_cognitive = ['cognitive decline','functional decline','cognitive']
    keywords_spinal_cord_injury = ["spinal cord injury"]
    keywords_knee_osteoarthritis = ["knee osteoarthritis"]
    keyword_cancer = ['cancer']
    keywords_mental_health_disorder = ["anxiety","obsessive-compulsive",'emotional regulation']

    keywords_none = ["none", "not", "nan"]

    if any(keyword in pathology for keyword in keywords_fibromyalgia):
        return "fibromyalgia"

    elif any(keyword in pathology for keyword in keywords_cognitive):
        return "cognitive decline"

    elif any(keyword in pathology for keyword in keywords_spinal_cord_injury):
        return "spinal cord injury"

    elif any(keyword in pathology for keyword in keywords_mental_health_disorder):
        return "mental-health disorder"

    elif any(keyword in pathology for keyword in keywords_schizophrenia):
        return "schizophrenia"

    elif any(keyword in pathology for keyword in keywords_depression):
        return "depression"

    elif any(keyword in pathology for keyword in keywords_knee_osteoarthritis):
        return "knee osteoarthritis"

    elif any(keyword in pathology for keyword in keyword_cancer):
        return "cancer"

    elif any(keyword in pathology for keyword in keywords_pain):
        return 'pain'

    elif any(keyword in pathology for keyword in keywords_stroke):
        return "stroke"

    elif any(keyword in pathology for keyword in keywords_none):
        return 'Not specified'

    else:
        return 'others' #pathology


In [ ]:
def standarize_symptom(symptom):
    symptom = str(symptom).lower()

    keywords_aphasia = ["aphasia", "post-stroke aphasia", "chronic aphasia"]
    keyword_motor = ['hemiparesis','hemiparetic','hemiplegic','paretic','paresis','apraxia','apraxic','spasticity','chronic stroke', 'cerebral stroke','ischemic stroke', 'acute ischemic stroke','catatonia']
    keywords_chronic_pain = ["chronic pain"]
    keywords_neuropathic = ["neuropathic pain"]
    keywords_depression = ["depression", 'depressive','treatment-resistant depression', 'treatment-resistant major depression','major depression','depression in','post-stroke depression']
    keywords_fibromyalgia  = ['fibromyalgia']
    keywords_schizophrenia = ['schizophrenia']
    keyword_cognitive_function = ["cognitive decline", "cognitive impairment",'functional decline','cognitive']

    keyword_others = ["dysphagia", "swallowing function", "post-stroke dysphagia","post-stroke fatigue", "fatigue","spatial neglect", "hemispatial neglect",'hyperalgesia','arthralgia','neuralgia','musculoskeletal pain',"abdominal/pelvic pain", "abdominal pain", 'pelvic pain',"low back pain",'lower back','complex regional pain syndrome', 'complex regional','pain perception','pain control', 'pain processing','pain modulation','shoulder pain','limb pain','restless legs syndrome','temporomandibular','temporo-mandibular','hemianopsia','hemianopia','migraine','trigeminal neuralgia','headache','pain syndromes', 'knee osteoarthritis', 'experimental pressure pain thresholds', 'acute middle cerebral artery stroke',"parkinson's disease	",'head and neck cancer', 'obesity']
    keywords_none = ["none", "not", 'nan', 'not specified']


    if any(keyword in symptom for keyword in keywords_aphasia):
        return "aphasia"

    elif any(keyword in symptom for keyword in keywords_chronic_pain):
        return "chronic pain"

    elif any(keyword in symptom for keyword in keywords_neuropathic):
        return "neuropathic pain"

    elif any(keyword in symptom for keyword in keywords_depression):
        return "depression"

    elif any(keyword in symptom for keyword in keywords_fibromyalgia):
        return "fibromyalgia"

    elif any(keyword in symptom for keyword in keywords_schizophrenia):
        return 'complex symptoms associated with schizophrenia'

    elif 'pain' in symptom:
        if 'chronic' in symptom:
            return 'chronic pain'
        else:
          return 'others'

    elif any(keyword in symptom for keyword in keyword_cognitive_function):
        return "cognitive function"

    elif any(keyword in symptom for keyword in keyword_motor):
        return "motor"

    elif any(keyword in symptom for keyword in keywords_none):
        return 'Not specified'

    else:
        return 'others' #symptom

In [ ]:
def standarize_electrodes(electrode):
    electrode = str(electrode).lower()

    keywords_C3_C4 = [keyword.lower() for keyword in ["C3", "C4", "M1",'Primary Motor Cortex','S1','Primary Somatosensory Cortex', 'motor cortex','somatosensory cortex', 'central region','Primary sensory cortex','FCC3h','C3h','FCC3', 'FCC4h','C4h','FCC4','motor']]
    keywords_Fp1_Fp2 = [keyword.lower() for keyword in ["Fp1", "Fp2", "Frontopolar",'PFC','Prefrontal','AFp7h','AFp5h','AFp3h','Fp1h','AFp7','AFp5','AFp3', 'AFp4h','AFp6h','AFp8h','Fp2h','AFp4','AFp6','AFp8']]
    keywords_F7_F8 = [keyword.lower() for keyword in ["F7", "F8",'Frontal',"frontal cortex", 'Broca','frontal operculum','FFT7h','F7h','AFF7','FFT7','FFT8h','F8h','FFT8','language']]
    keywords_F3_F4 = [keyword.lower() for keyword in ["F3", "F4", "dorsolateral prefrontal cortex", 'Dorsolateral prefrontal cortex', 'DLPFC','AF3','AFF5h','AFF3h','F5h','AFF3', 'AFF4h','AFF6h','F4h','F6h','AFF4']]
    keywords_T3_T4 = [keyword.lower() for keyword in ["T3", "T4", "temporal", 'Temporal','auditory cortex',"T7", "T8",'FSI','TTP7h','T7h','FTT7','TTP7','TTP8h','T8h','TTP8']]
    keywords_P3_P4 = [keyword.lower() for keyword in ["P3", "P4", "parietal", "Parietal",'P1','CPP3h','P3h','CPP3','P2','CPP4h','P4h','CPP2']]
    keywords_O1_O2 = [keyword.lower() for keyword in ["O1", "O2", "Occipital", 'occipital','V1','Primary Visual Cortex','PO3','PO1','POO5h','POO3h','PO5h','PO3h','POO3','POO1','PO2','PO4','POO4h','PO2h','PO4h','POO2','POO4','inion']]
    keywords_T5_T6  = [keyword.lower() for keyword in ["T5", "T6"]]
    keywords_A1_A2  = [keyword.lower() for keyword in ["A1", "A2",'Left Earlobe','Right Earlobe']]
    keywords_FpZ  = [keyword.lower() for keyword in ["FpZ", "Frontopolar Zero", "frontopolar zero", 'Frontopolar zero']]
    keywords_Fz  = [keyword.lower() for keyword in ["Fz", "Frontal Zero", "frontal Zero", 'Frontal zero', 'AFF1h','F1h','AFFz']]
    keywords_Oz  = [keyword.lower() for keyword in ["Oz", "Occipital Zero", "occipital zero", 'Occipital zero']]
    keywords_Cz  = [keyword.lower() for keyword in ["Cz", "Central Zero", "central zero", 'Central zero','FCC1h','C1h','FCCz']]
    keywords_supraorbital  = [keyword.lower() for keyword in ["supraorbital",'supra-orbital',"forehead above the orbit", "above the orbit", 'orbit','SO','over the left eyebrow']]
    keywords_anatomical  = [keyword.lower() for keyword in ["neck", "shoulder", "cerebellum",'cerebellar','leg','mylohyoid','deltoid','column']]
    keywords_none  = [keyword.lower() for keyword in ["none", "not specified",'nan']]

    if any(keyword in electrode for keyword in keywords_C3_C4):
        return "C3/C4"
    elif any(keyword in electrode for keyword in keywords_Fp1_Fp2):
        return "Fp1/Fp2"
    elif any(keyword in electrode for keyword in keywords_F7_F8):
        return "F7/F8"
    elif any(keyword in electrode for keyword in keywords_F3_F4):
        return "F3/F4"
    elif any(keyword in electrode for keyword in keywords_T3_T4):
        return "T3/T4"
    elif any(keyword in electrode for keyword in keywords_P3_P4):
        return "P3/P4"
    elif any(keyword in electrode for keyword in keywords_O1_O2):
        return "O1/O2"
    elif any(keyword in electrode for keyword in keywords_T5_T6):
        return "T5/T6"
    elif any(keyword in electrode for keyword in keywords_A1_A2):
        return "A1/A2"
    elif any(keyword in electrode for keyword in keywords_FpZ):
        return "FpZ"
    elif any(keyword in electrode for keyword in keywords_Fz):
        return "Fz"
    elif any(keyword in electrode for keyword in keywords_Oz):
        return "Oz"
    elif any(keyword in electrode for keyword in keywords_Cz):
        return "Cz"
    elif any(keyword in electrode for keyword in keywords_supraorbital):
        return "Supraorbital region"
    elif any(keyword in electrode for keyword in keywords_anatomical):
        return "others"
    elif any(keyword in electrode for keyword in keywords_none):
        return 'Not specified'
    else:
        return 'others' #electrode

In [ ]:
## LAST VERSION   75TH PERCENTILE

# Definir las palabras clave asociadas a cada grado de evidencia
grados_evidencia = {
    'A': ['placebo-controlled', 'sham', 'meta-analytic', 'double-blinded','randomized', 'controlled', 'crossover', 'cross-over', 'cross-over', 'within-subject', 'multicenter', 'meta-analysis', 'triple-blind', 'double-blind', 'double blinding', 'sham-controlled', 'rct', 'randomly', 'cross-over', 'randomised','review'],
    'B': ['cohort', 'case-control', 'single-blind', 'retrospective', 'case', 'prospective', 'cohort', 'single-blinded', 'single-blind', 'single blind'],
    'C': ['observational', 'longitudinal', 'non-experimental', 'cross-sectional', 'individual', 'responses', 'historical', 'open-label naturalistic', 'open-label feasibility', 'open label', 'unblinded', 'open-label', 'open', 'case series', 'observational', 'non-randomized', 'non-blinded', 'non-sham-controlled', 'feasibility', 'pilot',' open-label', 'proof-of-principle', 'proof of concept', 'safety'],
    'D': ['opinion', 'descriptive', 'animal', 'simulation-based', 'report' , 'single', 'case', 'simulation', 'computational', 'simulation-based', 'single-participant']
}

# Función para asignar el grado de evidencia a una palabra clave
def asignar_grado_evidencia(palabra):
    for grado, palabras_clave in grados_evidencia.items():
        if any(palabra.lower() == palabra_clave.lower() for palabra_clave in palabras_clave):
          return f'Grado {grado}'
    return None

def evidence_standarization(df):

    df['Evidence_level'] = 0

    ## Linea de codigo para calcular el percentil 75 de todos los N_samples
    # Ensure the 'N_sample' column is numeric
    df['N_sample'] = pd.to_numeric(df['N_sample'], errors='coerce')

    # Compute the 75th percentile of the 'N_sample' column
    percentile_75th = df['N_sample'].quantile(0.75)
    print('el percentil 75 es', percentile_75th)

    # Para cada registro
    for indice, fila in df.iterrows():

        # Contador de grados --> ranking
        contador_grados = {'A': 0, 'B': 0, 'C': 0, 'D': 0}

        evidencia = fila['Evidence']
        # Si la evidencia no está en blanco
        if isinstance(evidencia, str):
            # Dividirla en palabras clave individuales
            palabras_clave = [palabra.strip() for subcadena in evidencia.split(',') for palabra in subcadena.split(' ')]

            # Asignamos el grado de evidencia para cada palabra clave
            grados = [asignar_grado_evidencia(palabra) for palabra in palabras_clave]
            # Eliminamos los valores None de la lista
            grados = [grado for grado in grados if grado is not None]

            # Actualizamos el contador de grados
            for grado in grados:
                contador_grados[grado.split()[1]] += 1

            # Si se encontró al menos un grado de evidencia, asignamos el maximo a Evidence
            if grados:
                max_grado = max(contador_grados, key=contador_grados.get)
                df.at[indice, 'Evidence'] = max_grado
                value = contador_grados.get(max_grado)
                df.at[indice, 'Evidence_level'] = value

                # PUNTAJE POR SAMPLE (el threshold de los 80 primeros papers es 58.5, cambialo con el de todos)
                sample_size = fila['N_sample']
                #if int(sample_size) >= percentile_75th:
                if sample_size >= percentile_75th:  # 50.5
                  df.at[indice, 'Evidence_level'] += 1

            else:
                #return 'others'# Si no se encontraron grados de evidencia, mantener el valor original
                continue
    return df

In [ ]:
def standarize_yes_no(effectiveness):

    effectiveness = str(effectiveness).lower()
    keywords_yes  = ["yes", "y"]

    keywords_no  = ["no", "n"]

    keywords_none  = ["none", "not specified",'nan']

    if any(keyword in effectiveness for keyword in keywords_yes):
        return "YES"

    elif any(keyword in effectiveness for keyword in keywords_no):
        return "NO"

    elif any(keyword in effectiveness for keyword in keywords_none):
        return "Not specified"

    else:
        return effectiveness

In [ ]:
import re
def standardize_age(age):
    # Convert the age to a string and make it lowercase
    age = str(age).lower()
    keywords_18 = [
        "18 years and older", "18 years or older", '18 years old and above', "18+",
        'over 18', 'older than 18', '> 18', '>18', '≥ 18', 'greater than 18'
    ]
    keywords_none = ["none", "not", 'nan']

    keywords_mean = ["mean",'sd']

    keywords_weeks = ["week",'weeks']


    age = age.replace('to', '-')

    if any(keyword in age for keyword in keywords_18):
        return '18+'
    if any(keyword in age for keyword in keywords_weeks):
        return '0-2'

    if any(keyword in age for keyword in keywords_none):
        return 'Not specified'

    if any(keyword in age for keyword in keywords_mean):
        age = re.sub(r'\b(years|years-old).*', '', age)
        age = re.sub(r'\s*.*?', '', age)
        try:
            age_value = float(age)
            return f"{int(age_value)}"
        except ValueError:
            pass

    # Handle '±' for age ranges
    match = re.match(r"(\d+\.?\d*) ± (\d+\.?\d*)", age)
    if match:
        mean_age = float(match.group(1))
        range_age = float(match.group(2))
        min_age = mean_age - range_age
        max_age = mean_age + range_age
        return f"{int(min_age)}-{int(max_age)}"

    # Handle ranges with '-' and replace whitespace
    age = re.sub(r'\s*-\s*', '-', age)

    return age

In [ ]:
KEYWORDS = {
    "motor": ["motor", "limb", "gait", "rehabilitation", "hemiparesis", "hemiparetic", "paretic", "paresis", "movement", "arm function", "motor cortex", "standing balance", "foot positions", "walking", "muscle", "exercise", "training", "physiotherap"],
    #"hallucinations" : ['hallucinations','psychosis'],
    "cognitive function": ["cognitive function", "learning",'cognitive','cognition','attentional impairments','execute functions'],
    "chronic pain": ["chronic pain"],
    "neuropathic pain": ["neuropathic pain"],
    "fibromyalgia": ["fibromyalgia"],
}

# Keywords specific to fibromyalgia
FIBROMYALGIA_SUBKEYWORDS = {
    "pain associated with fibromyalgia": ["pain"],
    "fatigue associated with fibromyalgia": ["fatigue"]
}

def standarize_motor(symptom, original_value):
    symptom = str(symptom).lower()

    for category, keywords in KEYWORDS.items():
        if any(keyword in symptom for keyword in keywords):
            if category == "fibromyalgia":
                for subcategory, subkeywords in FIBROMYALGIA_SUBKEYWORDS.items():
                    if any(subkeyword in symptom for subkeyword in subkeywords):
                        return subcategory
                return 'complex symptoms associated with fibromyalgia'
            return category

    return original_value

In [ ]:
def classify_age_ranges(df):
    categories = {
        'Kid_0_5':(0,5),
        'Youth_6_17':(6,17),
        'Adult_18_31': (18, 31),
        'Adult_32_59': (32, 59),
        'Elderly_60_100': (60, 100),
    }

    # Initialize new columns for categories with 0
    for category in categories:
        df[category] = 0

    # Initialize 'No_coincidence' column
    df['No_coincidence'] = ''

    # Function to classify age range for a single entry
    def classify_single_age_range(idx, age_range):
        try:
            if age_range == 'Not specified':
                return

            age_range = age_range.replace('+', '-100')
            age_range_parts = age_range.split('-')
            age_min = int(age_range_parts[0])
            age_max = int(age_range_parts[1]) if len(age_range_parts) > 1 else age_min

            range_sample = range(age_min, age_max + 1)

            for category, (cat_min, cat_max) in categories.items():
                range_category = range(cat_min, cat_max + 1)

                if any(age in range_category for age in range_sample):
                    df.loc[idx, category] = 1

        except:
            # If the format is incorrect, keep the original value
            df.loc[idx, 'No_coincidence'] = age_range

    # Apply function to classify age ranges
    for idx in df.index:
        classify_single_age_range(idx, df.loc[idx, 'Age'])

    return df

In [ ]:
import numpy as np
def clean_males_column(value):
    if isinstance(value, str):
        value = value.strip()
        if value.endswith('%'):
            return float(value.rstrip('%'))
        elif ',' in value:
            numbers = value.split(',')
            try:
                numbers = [float(num.strip()) for num in numbers]
                return np.mean(numbers)
            except ValueError:
                return np.nan  # If conversion fails, return NaN
        else:
            return np.nan  # For other strings, return NaN
    return value

In [ ]:
def standardize_gender(df):
    df['No_gender'] = float(0)

    def compute_from_sample(gender, sample_value):
        # Convert gender to string
        gender = str(gender)

        # Handle 'Not specified' cases
        if gender == 'Not specified':
            return 0.0

        # Remove percentage sign and convert to float
        if '%' in gender:
            gender = gender.replace('%', '')
            try:
                gender = float(gender)
                return gender / 100
            except ValueError:
                return 0.0

        # Handle cases with commas
        if ',' in gender:
            # Remove words and keep only numbers and commas
            gender = re.sub(r'\s*,\s*', ',', gender)
            gender = re.sub(r'[^\d,]', '', gender)
            # Split the values by commas
            gender_split = gender.split(',')
            value = 0
            for part in gender_split:
                try:
                    number = float(part)
                    value += number
                except ValueError:
                    pass
            return value / sample_value

        # Handle simple numeric cases
        try:
            gender = float(gender)
            return gender / sample_value
        except ValueError:
            return 0.0

    # Loop through each row and standardize the 'Males' and 'Females' columns
    for idx in df.index:
        sample_value = df.loc[idx, 'N_sample']
        if not pd.isna(sample_value):
            try:
                sample_value = float(sample_value)
                df.at[idx, 'Males'] = compute_from_sample(df.loc[idx, 'Males'], sample_value)
                df.at[idx, 'Females'] = compute_from_sample(df.loc[idx, 'Females'], sample_value)

                if df.loc[idx, 'Males'] == 0.0 and df.loc[idx, 'Females'] == 0.0:
                    df.at[idx, 'No_gender'] = float(1)

            except ValueError:
                pass

    return df

In [ ]:
def standarize_origin(origin):
    origin = str(origin)

    keyword_america = ['USA','United States','American','Brazil','Argentina','Chile','Canada','Latin','Caribbean','Florida','Reagan']
    keyword_europe = ['France','Romania','Czechia','Spain','Italy','Italian','Poland','Netherlands','UK','United Kingdom','Denmark','Belgium','Czech Republic','Greece','Ireland','Serbia','Germany','Switzerland','British']
    keyword_asia = ['Asia','China','Korea','Japan','Korean','Chinese','Iran','India','Taiwan','Thailand','Indonesia','Hong Kong','Singapore','Thai','Saudi Arabia','Jordan','Pakistan']
    keyword_australia = ['Australia','New Zealand']
    keyword_africa = ['Africa','Egypt']

    keywords_none  = ["none", "not specified",'nan','Multiple countries','Global']

    if any(keyword in origin for keyword in keyword_europe):
        return "Europe"

    elif any(keyword in origin for keyword in keyword_america):
        return "America"

    elif any(keyword in origin for keyword in keyword_asia):
        return "Asia"

    elif any(keyword in origin for keyword in keyword_australia):
        return "Australia"

    elif any(keyword in origin for keyword in keyword_africa):
        return "Africa"

    elif any(keyword in origin for keyword in keywords_none):
        return 'Not specified'

    else:
        return origin

In [ ]:
def standardize_none(column):
    column = str(column).lower()

    keywords_none  = ["none", "not specified",'nan']

    if any(keyword in column for keyword in keywords_none):
        return 0.0 #'Not specified'
    else:
        try:
          return float(column)
        except:
          return column

In [ ]:
def clean_and_standardize_gender(df):
    def compute_from_sample(gender, sample_value):
        # Convert gender to string
        gender = str(gender)

        # Handle 'Not specified' cases
        if gender == 'Not specified':
            return 0.0

        # Remove percentage sign and convert to float
        if '%' in gender:
            gender = gender.replace('%', '')
            try:
                gender = float(gender)
                return gender / 100
            except ValueError:
                return 0.0

        # Handle cases with commas
        if ',' in gender:
            # Remove words and keep only numbers and commas
            gender = re.sub(r'\s*,\s*', ',', gender)
            gender = re.sub(r'[^\d,]', '', gender)
            # Split the values by commas
            gender_split = gender.split(',')
            value = 0
            for part in gender_split:
                try:
                    number = float(part)
                    value += number
                except ValueError:
                    pass
            return value / sample_value

        # Handle simple numeric cases
        try:
            gender = float(gender)
            return gender / sample_value
        except ValueError:
            return 0.0

    # Add 'No_gender' column if not present
    if 'No_gender' not in df.columns:
        df['No_gender'] = float(0)

    # Loop through each row and standardize the 'Males' and 'Females' columns
    for idx in df.index:
        sample_value = df.loc[idx, 'N_sample']
        if not pd.isna(sample_value):
            try:
                sample_value = float(sample_value)
                df.at[idx, 'Males'] = compute_from_sample(df.loc[idx, 'Males'], sample_value)
                df.at[idx, 'Females'] = compute_from_sample(df.loc[idx, 'Females'], sample_value)

                if df.loc[idx, 'Males'] == 0.0 and df.loc[idx, 'Females'] == 0.0:
                    df.at[idx, 'No_gender'] = float(1)

            except ValueError:
                pass

    return df

In [ ]:
def standarize_review(evidence,original_value):
    evidence = str(evidence).lower()
    keywords_review = ['review','systematic review','mini-review']
    if any(keyword in evidence for keyword in keywords_review):
        return "review"
    else:
        return original_value

In [ ]:
def sociodemo_standarization(df):
    df['Age'] = df['Age'].apply(standardize_age)
    df['Age'] = df['Age'].str.replace(r'\b(years|years-old).*', '', regex=True)
    df['Age'] = df['Age'].str.replace(r'\s*.*?', '', regex=True)
    df['Males'] = df['Males'].str.replace(r'\s*.*?', '', regex=True)
    df['Females'] = df['Females'].str.replace(r'\s*.*?', '', regex=True)
    df['Males'] = df['Males'].str.strip()
    df['Females'] = df['Females'].str.strip()
    df['Males'] = df['Males'].str.replace(r'\b(males|male).*', '', regex=True)
    df['Females'] = df['Females'].str.replace(r'\b(females|female).*', '', regex=True)
    df['N_sample'] = df['N_sample'].apply(lambda x: x if x == 'Not specified' else pd.Series(x).str.replace(r'(\b\w+\b)\s+(participants|patients|patient|survivors|chronic|post|stroke|people|subjects).*', r'\1', regex=True).iloc[0])
    df['Males'] = df['Males'].apply(standardize_none)
    df['Females'] = df['Females'].apply(standardize_none)
    df['Origin_standarized'] = df['Origin'].apply(standarize_origin)
    df = classify_age_ranges(df)
    df['Males'] = df['Males'].apply(clean_males_column)
    df['Females'] = df['Females'].apply(clean_males_column)
# Convert the cleaned 'Males' column to float
    df = clean_and_standardize_gender(df)
    df['Males'] = df['Males'].astype(float)
    df['Females'] = df['Females'].astype(float)
    return df

In [ ]:
def parameters_standarization(df):
    df['Current (mA)'] = df['Current (mA)'].apply(standardize_current)
    df['Modality'] = df['Modality'].apply(standarize_modality)
    df['Duration (min)'] = df['Duration (min)'].apply(standarize_duration)
    df['Times_per_day'] = df['Times_per_day'].apply(standarize_times_perday)
    df['Days_per_week'] = df['Days_per_week'].apply(standarize_days_perweek)
    df['Weeks'] = df['Weeks'].apply(standarize_weeks)
    df['Symptom_Keyword'] = df['Pathology'].apply(standarize_symptom)
    df['Symptom_Keyword'] = df.apply(lambda row: standarize_motor(row['Title'], row['Symptom_Keyword']), axis=1)
    df['Evidence'] = df.apply(lambda row: standarize_review(row['Title'], row['Evidence']), axis=1)
    df['Pathology_standarized'] = df['Pathology'].apply(standarize_pathology)
    df['Anode_standarized'] = df['Anode'].apply(standarize_electrodes)
    df['Cathode_standarized'] = df['Cathode'].apply(standarize_electrodes)
    df['Effectiveness'] = df['Effectiveness'].apply(standarize_yes_no)
    return df

In [ ]:
def parameters_cleaning(df):
    df['Duration'] = df['Duration'].str.replace(r'\bmin.*', '', regex=True)
    df['Current'] = df['Current'].str.replace(r'\b(mA|milliamperes|/mm2|milliamps).*', '', regex=True)
    df['Weeks'] = df['Weeks'].str.replace(r'\b(Week|Weeks|week|weeks).*', '', regex=True)
    df = df.rename(columns={'Current': 'Current (mA)', 'Duration': 'Duration (min)'})
    df['Anode'] = df['Anode'].fillna('None')
    df['Cathode'] = df['Cathode'].fillna('None')
    df_none_values_anode = df[df['Anode'] == 'None']
    df_none_values_cathode = df[df['Cathode'] == 'None']
    df = df.drop(df_none_values_anode.index)
    indices_to_drop_cathode = df_none_values_cathode.index.intersection(df.index)
    df = df.drop(indices_to_drop_cathode)
    return df

In [ ]:
def standarize_sessions_int(df):
  """
  Changes the information stored in the column 'Sessions' to integer type.
    - Eliminates the word 'session(s)'
    - Changes the value to integer type
  """
  keywords_5_days = ['five','-5','to 5']
  keywords_2_days = ['twice','two','to 2','-2']
  keywords_3_days  = ['three','-3','to 3']
  keywords_none  = ["none", "not",'nan']
  keywords_7_days  = ["seven",'-7','to 7']
  keywords_6_days  = ["six", 'to 6', '-6']
  keywords_1_days  = ["one","once", "single"]

  df_copy = df.copy()

  for index, row in df_copy.iterrows():
    sessions = row['Sessions']
    sessions = str(sessions)
    sessions = sessions.lower()

    if 'session' in sessions:
        if any(keyword in sessions for keyword in keywords_5_days):
            sessions = 5.0
            df_copy.at[index, 'Sessions'] = sessions

        elif any(keyword in sessions for keyword in keywords_2_days):
            sessions = 2.0
            df_copy.at[index, 'Sessions'] = sessions

        elif any(keyword in sessions for keyword in keywords_3_days):
            sessions = 3.0
            df_copy.at[index, 'Sessions'] = sessions

        elif any(keyword in sessions for keyword in keywords_none):
            sessions = 0.0
            df_copy.at[index, 'Sessions'] = sessions

        elif any(keyword in sessions for keyword in keywords_7_days):
            sessions = 7.0
            df_copy.at[index, 'Sessions'] = sessions

        elif any(keyword in sessions for keyword in keywords_6_days):
            sessions = 6.0
            df_copy.at[index, 'Sessions'] = sessions

        elif any(keyword in sessions for keyword in keywords_1_days):
            sessions = 1.0
            df_copy.at[index, 'Sessions'] = sessions

        else:
            try:
              sessions = sessions.replace('session', '')
              sessions = sessions.split()[0]
              sessions = float(sessions)
              df_copy.at[index, 'Sessions'] = sessions

            except ValueError:
              sessions = 0.0
              df_copy.at[index, 'Sessions'] = sessions
    else:
        try:
          sessions = float(sessions)
          df_copy.at[index, 'Sessions'] = sessions
        except ValueError:
          sessions = 0.0
          df_copy.at[index, 'Sessions'] = sessions

  return df_copy

In [ ]:
def standarize_periodicity_sessions(df):
  """
  Fills in the periodicity columns, when there is only 1 session.
  """

  for index, row in df.iterrows():
    periodicity_info = row['Periodicity_info']
    sessions = row['Sessions']

    if 'one' in periodicity_info.lower():
      periodicity_info = periodicity_info.replace('one', '1')

    if sessions == 0.0:

      if 'sessions' in periodicity_info.lower():
          match = re.search(r'(\d+)\s+(sessions)', periodicity_info.lower())
          if match:
              sessions = match.group(1)
              try:
                  sessions = float(sessions)
              except ValueError:
                  sessions = 0.0
              df.at[index, 'Sessions'] = sessions
          else:
              sessions = 0.0
              df.at[index, 'Sessions'] = sessions

      if 'session' in periodicity_info.lower():
          match = re.search(r'(\d+)\s+(session)', periodicity_info.lower())
          if match:
              sessions = match.group(1)
              try:
                  sessions = float(sessions)
              except ValueError:
                  sessions = 0.0
              df.at[index, 'Sessions'] = sessions
          else:
              sessions = 0.0
              df.at[index, 'Sessions'] = sessions
      else:
        sessions = 0.0
        df.at[index, 'Sessions'] = sessions

    if sessions == 1.0:
      times_per_day = 1.0
      days_per_week = 1.0
      weeks = 1.0

      df.at[index, 'Times_per_day'] = times_per_day
      df.at[index, 'Days_per_week'] = days_per_week
      df.at[index, 'Weeks'] = weeks
  return df

In [ ]:
def standarize_periodicity_info(df):

  df_copy = df.copy()

  for index, row in df_copy.iterrows():
    periodicity_info = row['Periodicity_info']
    periodicity_info = str(periodicity_info)
    periodicity_info = periodicity_info.lower()

    if 'one' in periodicity_info:
      periodicity_info = periodicity_info.replace('one', '1')

    elif 'two' in periodicity_info:
      periodicity_info = periodicity_info.replace('two', '2')

    elif 'three' in periodicity_info:
      periodicity_info = periodicity_info.replace('three', '3')

    elif 'four' in periodicity_info:
      periodicity_info = periodicity_info.replace('four', '4')

    elif 'five' in periodicity_info:
      periodicity_info = periodicity_info.replace('five', '5')

    elif 'six' in periodicity_info:
      periodicity_info = periodicity_info.replace('six', '6')

    elif 'seven' in periodicity_info:
      periodicity_info = periodicity_info.replace('seven', '7')

    elif 'eight' in periodicity_info:
      periodicity_info = periodicity_info.replace('eight', '8')

    elif 'nine' in periodicity_info:
      periodicity_info = periodicity_info.replace('nine', '9')

    elif 'ten' in periodicity_info:
      periodicity_info = periodicity_info.replace('ten', '10')

    elif 'eleven' in periodicity_info:
      periodicity_info = periodicity_info.replace('eleven', '11')

    elif 'twelve' in periodicity_info:
      periodicity_info = periodicity_info.replace('twelve', '12')

    elif 'thirteen' in periodicity_info:
      periodicity_info = periodicity_info.replace('thirteen', '13')

    elif 'twenty' in periodicity_info:
      periodicity_info = periodicity_info.replace('twenty', '20')

    elif 'thirty' in periodicity_info:
      periodicity_info = periodicity_info.replace('thirty', '30')

    elif 'forty' in periodicity_info:
      periodicity_info = periodicity_info.replace('forty', '40')

    elif 'fifty' in periodicity_info:
      periodicity_info = periodicity_info.replace('fifty', '50')

    df_copy.at[index, 'Periodicity_info'] = periodicity_info

  return df_copy

In [ ]:
def standarize_periodicity_general(df):
  """
  Fills in the periodicity columns, following several rules.
  """

  df_standarized_copy = standarize_sessions_int(df)
  df_standarized_copy = standarize_periodicity_info(df_standarized_copy)
  df_standarized_copy = standarize_periodicity_sessions(df_standarized_copy)

  for index, row in df_standarized_copy.iterrows():
      periodicity_info = row['Periodicity_info']
      sessions = row['Sessions']
      times_per_day = row['Times_per_day']
      times_per_day = str(times_per_day)
      days_per_week = row['Days_per_week']
      days_per_week = str(days_per_week)
      weeks = row['Weeks']
      weeks = str(weeks)

      if periodicity_info.lower() == 'not specified':
        continue

      else:

        if row['Times_per_day'] == '0.0':
            if 'once' in periodicity_info.lower():
                times_per_day = 1.0
            elif 'twice' in periodicity_info.lower():
                times_per_day = 2.0
            elif 'daily' in periodicity_info.lower():
                times_per_day = 1.0
            else:
                match = re.search(r'(\d+)\s+(stimulation|stimulations|sessions|times|session|time)\s+(per|each|a)\s+(day|visit)', periodicity_info.lower())
                if match:
                    times_per_day = match.group(1)
                else:
                    times_per_day = 1.0

            df_standarized_copy.at[index, 'Times_per_day'] = times_per_day

        if row['Weeks'] == '0.0':
            match = re.search(r'(\d+)\s+(weeks|- weeks)?', periodicity_info.lower())
            if match:
                weeks = match.group(1)
            elif 'week' in periodicity_info.lower():
                weeks = 1.0
            else:
                weeks = 0.0
            df_standarized_copy.at[index, 'Weeks'] = weeks

        if row['Weeks'] != '0.0' and 'weeks' in periodicity_info.lower():
            match = re.search(r'(\d+)\s+weeks?', periodicity_info.lower())
            if match:
                weeks = match.group(1)
            else:
                weeks = row['Weeks']
            df_standarized_copy.at[index, 'Weeks'] = weeks

        if row['Days_per_week'] == '0.0':
            match = re.search(r'(\d+)\s+(sessions|times|days|session|time|day|visit)\s+(per|each|a)?\s*week', periodicity_info.lower())
            if match:
                days_per_week = match.group(1)

            elif 'consecutive' in periodicity_info.lower():
                match = re.search(r'(\d+)\s+(consecutive)\s+(days|sessions|times)', periodicity_info.lower())
                if match:
                    days_per_week = match.group(1)

            elif 'daily' in periodicity_info.lower():
                if sessions <= 5.0:
                    days_per_week = sessions
                else:
                    while sessions > 5.0:
                        sessions -= 5.0
                    days_per_week = f'{sessions}-5'

            elif 'alternate days' in periodicity_info.lower():
                if sessions <= 3.0:
                    days_per_week = sessions
                else:
                    while sessions > 3.0:
                        sessions -= 3.0
                    days_per_week = f'{sessions}-3'

            elif '48h appart' in periodicity_info.lower():
                if sessions <= 2.0:
                    days_per_week = sessions
                else:
                    while sessions > 2.0:
                        sessions -= 2.0
                    days_per_week = f'{sessions}-2'

            elif 'weekly' in periodicity_info.lower():
                days_per_week = 1.0

            else:
              if row['Weeks'] != '0.0' and row['Sessions'] != 'Not specified':
                weeks = float(weeks)
                sessions = float(sessions)
                days_per_week = sessions/weeks
                days_per_week = round(days_per_week)
                if days_per_week > 5.0:
                  days_per_week = 5.0
                else:
                  days_per_week = days_per_week
              else:
                days_per_week = 0.0

            df_standarized_copy.at[index, 'Days_per_week'] = days_per_week
        else:
          continue

  return df_standarized_copy

In [ ]:
def standarization (df):
    df = parameters_cleaning(df)
    df = parameters_standarization(df)
    df = standarize_periodicity_general(df)
    df = sociodemo_standarization(df)
    df = evidence_standarization(df)

    required_columns = ['corpusid', 'Year', 'Title', 'DOI', 'Current (mA)', 'Duration (min)', 'Periodicity_info',
                        'Sessions', 'Times_per_day', 'Days_per_week', 'Weeks', 'Cathode', 'Cathode_standarized',
                        'Anode', 'Anode_standarized', 'Modality', 'Pathology',
                        'Pathology_standarized', 'Symptom_Keyword', 'Evidence', 'Evidence_level', 'Effectiveness',
                        'Males', 'Females', 'No_gender', 'N_sample', 'Age', 'Kid_0_5', 'Youth_6_17', 'Adult_18_31',
                        'Adult_32_59', 'Elderly_60_100', 'Origin', 'Origin_standarized', 'text_url']

    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        print("Missing columns: ", missing_columns)
        raise KeyError(f"Missing columns in dataframe: {missing_columns}")

    df_final = df[required_columns]
    return df_final

In [ ]:
# Prueba
df_protocolo_parameters_prueba_copy = df_protocolo_parameters_prueba.copy()

In [ ]:
df_standarized = standarization(df_protocolo_parameters_prueba_copy)
df_standarized

el percentil 75 es 24.5


,corpusid,Year,Title,DOI,Current (mA),Duration (min),Periodicity_info,Sessions,Times_per_day,Days_per_week,...,No_gender,N_sample,Age,Kid_0_5,Youth_6_17,Adult_18_31,Adult_32_59,Elderly_60_100,Origin,Origin_standarized
0,261078356,2023,Efficacy and safety of transcranial direct cur...,10.1186/s12888-023-05112-0,0.0,0.0,"Weekdays of the first, third, and fifth weeks",Not specified,1.0,5.0,...,0.0,38.0,,0,0,0,0,0,China,Asia
1,235701129,2021,Transcranial Direct Current Stimulation in Str...,10.3389/fpain.2021.696547,0.0,0.0,Not specified,Not specified,0.0,0.0,...,0.0,NaN,,0,0,0,0,0,United States,America
2,249709833,2023,Multifocal Transcranial Direct Current Stimula...,10.1371/journal.pone.0270047,0.0,20.0,"within-subjects, cross-over",three experimental sessions,0.0,0.0,...,0.0,18.0,,0,0,0,0,0,Belgium,Europe
3,52058862,2023,Combined transcranial direct current stimulati...,10.2340/16501977-2379,0.0,20.0,single session,1 session,0.0,0.0,...,0.0,12.0,,0,0,0,0,0,USA,America
4,269142473,2024,The Application of tDCS to Treat Pain and Psyc...,10.1155/2024/6344925,0.0,0.0,Not specified,Not specified,0.0,0.0,...,0.0,NaN,,0,0,0,0,0,Italy,Europe
5,233820197,2021,The immediate effect of transcranial direct cu...,Not specified,2.0,20.0,one session,3 types of protocol,0.0,0.0,...,0.0,20.0,,0,0,0,0,0,Brazil,America


In [ ]:
## Saving in the Data Lake as df_BBDD_original (as csv and Excel)
output_parquet_path_4= dtset_dir_parquet.joinpath("df_standarized_prueba")
df_standarized.to_csv(output_parquet_path_4, index=False)
print(f"DataFrame saved as Parquet file at: {output_parquet_path_4}")

DataFrame saved as Parquet file at: /content/drive/MyDrive/Data Lake IONClinics/S2ORC completo/spark/df_standarized_prueba


## **PROCESO 2: Actualización y Registro de cambios**

En este proceso se realiza una revisión y actualización del conjunto de artículos relacionados con tDCS. El objetivo es incorporar nuevos estudios científicos relevantes que hayan sido publicados recientemente, así como corregir o mejorar filtrados anteriores si fuera necesario. Esta etapa garantiza que la base de datos se mantenga actualizada y alineada con la evidencia más reciente disponible.

### **Actualización de la función de reconocimiento de entidades con GPT (NER)**

En esta etapa se actualiza la función de reconocimiento de entidades nombradas (NER) basada en GPT, ampliando su capacidad para extraer información más específica y estructurada. El proceso incluye:

1. **Diseño de nuevos prompts**, adaptados a distintos tipos de información: un prompt general, uno específico para periodicidad del tratamiento y otro para datos sociodemográficos.

2. **Extracción en formato JSON**, que permite mantener una estructura organizada y fácilmente interpretable.

3. **Generación de un DataFrame** con los datos extraídos, que se integra al flujo de análisis para su posterior procesamiento por el sistema de recomendación.

Recupera la base de datos original ya procesada y sin procesar, para utilizarla como contexto en la consulta GPT.

In [ ]:
df_BBDD_nonprocessed_example = pd.read_csv(path_df_BBDD_nonprocessed_example_csv)  # before GPT
df_BBDD_example = pd.read_csv(path_df_BBDD_example) # after GPT

**1) DISEÑO DE PROMPT**

**PROMPT GENERAL**

En este proceso se utiliza un prompt diseñado específicamente para extraer, a partir de la base de datos original, todos los campos relevantes del protocolo, exceptuando aquellos relacionados con la periodicidad del tratamiento y la información sociodemográfica. Este prompt permite obtener de manera estructurada los principales parámetros técnicos y clínicos de cada protocolo, manteniendo la coherencia con el formato general del sistema.

In [ ]:
client = OpenAI(api_key = GPT_api_key)

def NER_2_updates(text, previous_text, previous_df):
    try:
        prompt = f"""Previously, from this text dataframe: {previous_text}
        You created the following dataframe by extracting the information about the named entities related to tDCS interventions:
        {previous_df}.

        Now I want you, given this example, to please do the same for the following text dataframe: {text}
        """
        response = client.chat.completions.create(
            model="gpt-4-turbo",
            response_format={"type": "json_object"},
            messages=[
                {
                    "role": "system",
                    "content": "As a natural language processing expert who responds using JSON, your task is to extract named entities related to tDCS interventions, including: 1- Current intensity in milliamps (mA) used (e.g. 2mA, 1mA). 2- Duration in minutes (min) (e.g. 20 minutes, 30 minutes). 3- Any information about the electrode placement, preferably in 10-20 system locations (e.g., C4, F3). Focus on anode and cathode placements. 4- Stimulation modality (e.g., anodal tDCS, bilateral tDCS, high-definition tDCS). 5- Pathology being treated (e.g. aphasia, neuropathic pain). 6- Study/trial methodology design used (e.g., double-blinded, sham-controlled, RCT,review, meta-analysis). 7- A (YES/NO) whether the tDCS intervention showed experimental evidence of effectiveness or not. 8- The title of the research paper. 9- The doi of the research papaer. 10- The year that was published the research paper. If any information is not available, you must use 'Not specified' or an empty list."
                },
                {
                    "role": "assistant",
                    "content": "Your extracted entities should be formatted as follows: {'Year': ['year'],'Title': ['title'], 'DOI': ['doi'], 'Current': ['current_intensity'], 'Duration': ['duration'], 'Cathode': ['cathode_placement_10_20'], 'Anode': ['anode_placement_10_20'],'Modality': ['stimulation_modality'],'Pathology': ['pathology'],'Evidence': ['trial_design'], 'Effectiveness': ['effectiveness_yes_no']}. If any information is not available, please use 'Not specified' or an empty list."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0,
            top_p=1,
            frequency_penalty=0,
            presence_penalty=0
        )
        result = response.choices[0].message.content.strip(" \n")
        return result
    except Exception as e:
        print("Error in NER_2_updates:", e)
        return "{}"  # Devolver un JSON vacío en caso de error

**PROMPT DE PERIODICIDAD**

En este proceso se emplea un prompt específico para extraer únicamente la información relacionada con la periodicidad del tratamiento. A partir del DataFrame de contexto, se identifican y procesan las columnas correspondientes al número de sesiones por día, días por semana y duración total en semanas. Este enfoque permite capturar con mayor precisión la frecuencia y el esquema temporal de aplicación de los protocolos.

In [ ]:
client = OpenAI(api_key = GPT_api_key)

def NER_2_periodicity(text, previous_text, previous_df):

    previous_df = previous_df[['corpusid', 'Periodicity_info', 'Sessions', 'Times_per_day', 'Days_per_week', 'Weeks']]

    try:
        prompt = f"""Previously, from this text dataframe: {previous_text}
        You created the following dataframe by extracting the information about the named entities related to tDCS interventions:
        {previous_df}.

        Now I want you, given this example, to please do the same for the following text dataframe: {text}
        """
        response = client.chat.completions.create(
            model="gpt-4-turbo",
            response_format={"type": "json_object"},
            messages=[
                {
                    "role": "system",
                    "content": "As a natural language processing expert who responds using JSON, your task is to extract named entities related to tDCS interventions, including: Session details such as any information about the periodicity of the intervention (number of sessions, their distribution, how and when will they take place) (e.g. 15 daily sessions over 3 weeks, two times per day for 5 consecutive days),  number of sessions (e.g. 10 sessions, 20 sessions), times per day, days per week and weeks or months the sessions are administered. If any information is not available, you must use 'Not specified'."
                },
                {
                    "role": "assistant",
                    "content": "Your extracted entities must be formatted as follows: {'Periodicity_info': ['periodicity_info'],'Sessions': ['number_sessions'], 'Times per day': ['times_per_day'], 'Days per week': ['days_per_week'], 'Weeks': ['weeks_months']}. If any information is not available, please use ['Not specified']. Remember that the information you extract must always be inserted in the json between brackets."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0,
            top_p=1,
            frequency_penalty=0,
            presence_penalty=0
        )
        result = response.choices[0].message.content.strip(" \n")
        return result
    except Exception as e:
        print("Error in NER_2_periodicity:", e)
        return "{}"  # Devolver un JSON vacío en caso de error

**PROMPT SOCIO-DEMOGRÁFICO**

Tomamos del marco de datos contextual las columnas relacionadas con la información sociodemográfica del tratamiento.

In [ ]:
client = OpenAI(api_key = GPT_api_key)

def NER_2_sociodemo(text, previous_text, previous_df):

    previous_df = previous_df[['corpusid', 'Age', 'Males', 'Females', 'N_sample', 'Origin']]

    try:
        prompt = f"""Previously, from this text dataframe: {previous_text}
        You created the following dataframe by extracting the information about the named entities related to tDCS interventions:
        {previous_df}.

        Now I want you, given this example, to please do the same for the following text dataframe: {text}
        """
        response = client.chat.completions.create(
            model="gpt-4-turbo",
            response_format={"type": "json_object"},
            messages=[
                {
                    "role": "system",
                    "content": "As a natural language processing expert who responds using JSON, your task is to extract named entities related to tDCS interventions, including: 1- Age range of the sample used in the intervention. It can be expressed in ranges, numbers or even words (e.g. 18-70 years, greater than 18, aged 60, seventy-two years). 2- Gender distribution of the sample (number of males and number of females). 3- Number of people used as samples, usually numbers preceding the words 'individuals', 'patients' or 'participants' (e.g. 120 individuals, 30 patients, 60 participants, eight participants). 4- Country or origin of the sample (e.g. Italy, Korea, Spain). If any information is not available, you must use 'Not specified'."
                },
                {
                    "role": "assistant",
                    "content": "Your extracted entities must be formatted as follows: {'Age': ['age'], 'Males': ['males']'Females': ['females'], 'N_sample': ['n_sample'],'Origin': ['origin']}. If any information is not available, please use ['Not specified']. Remember that the information you extract must always be inserted in the json between brackets."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0,
            top_p=1,
            frequency_penalty=0,
            presence_penalty=0
        )
        result = response.choices[0].message.content.strip(" \n")
        return result
    except Exception as e:
        print("Error in NER_2_sociodemo:", e)
        return "{}"  # Devolver un JSON vacío en caso de error

**2) EXTRACTOR DE FORMATO JSON**

In [ ]:
def extract_info_from_json(json_str):
    try:
        data = json.loads(json_str)

        extracted_info = {
            "Year": data.get("Year", [None])[0],
            "Title": data.get("Title", [None])[0],
            "DOI": data.get("DOI", [None])[0],
            "Current": data.get("Current", [None])[0],
            "Duration": data.get("Duration", [None])[0],
            "Cathode": data.get("Cathode", [None])[0],
            "Anode": data.get("Anode", [None])[0],
            "Modality": data.get("Modality", [None])[0],
            "Pathology": data.get("Pathology", [None])[0],
            "Evidence": data.get("Evidence", [None])[0],
            "Effectiveness": data.get("Effectiveness", [None])[0]
        }

        return extracted_info
    except Exception as e:
        print("Error:", e)
        return None

In [ ]:
def extract_info_from_json_periodicity(json_str):
    try:
        data = json.loads(json_str)

        extracted_info = {
            "Periodicity_info": data.get("Periodicity_info", [None])[0],
            "Sessions": data.get("Sessions", [None])[0],
            "Times_per_day": data.get("Times per day", [None])[0],
            "Days_per_week": data.get("Days per week", [None])[0],
            "Weeks": data.get("Weeks", [None])[0]
        }

        return extracted_info
    except Exception as e:
        print("Error:", e)
        return None

In [ ]:
def extract_info_from_json_sociodemo(json_str):
    try:
        data = json.loads(json_str)

        extracted_info = {
            "Age": data.get("Age", [None])[0] if data.get("Age") else None,
            "Males": data.get("Males", [None])[0] if data.get("Males") else None,
            "Females": data.get("Females", [None])[0] if data.get("Females") else None,
            "N_sample": data.get("N_sample", [None])[0] if data.get("N_sample") else None,
            "Origin": data.get("Origin", [None])[0] if data.get("Origin") else None
        }

        return extracted_info
    except Exception as e:
        print("Error:", e)
        return None

**3) GENERACIÓN DEL DATAFRAME**

In [ ]:
def combine_extracted_info(dict_rest, dict_periodicity, dict_sociodemo):
    try:

        combined_info = {
            "Year": dict_rest['Year'],
            "Title": dict_rest['Title'],
            "DOI": dict_rest['DOI'],
            "Current": dict_rest['Current'],
            "Duration": dict_rest['Duration'],
            "Periodicity_info": dict_periodicity['Periodicity_info'],
            "Sessions": dict_periodicity['Sessions'],
            "Times_per_day": dict_periodicity['Times_per_day'],
            "Days_per_week": dict_periodicity['Days_per_week'],
            "Weeks": dict_periodicity['Weeks'],
            "Cathode": dict_rest['Cathode'],
            "Anode": dict_rest['Anode'],
            "Modality": dict_rest['Modality'],
            "Pathology": dict_rest['Pathology'],
            "Evidence": dict_rest['Evidence'],
            "Effectiveness": dict_rest['Effectiveness'],
            "Age": dict_sociodemo['Age'],
            "Males": dict_sociodemo['Males'],
            "Females": dict_sociodemo['Females'],
            "N_sample": dict_sociodemo['N_sample'],
            "Origin": dict_sociodemo['Origin']
        }
        return combined_info

    except Exception as e:
        print("Error:", e)
        return None

In [ ]:
def parameter_extraction(df):
    df_protocolo_parameters = []
    references_keyword = "Acknowledgments"

    df_size = df.count()
    df_selected = df.head(df_size)
    #df_selected = df.head(50)  ## We use this line if we just want to prove the prompt for the first 50 papers in the list

    for record in df_selected:
        text = str(record.text.replace('\n', ' '))
        corpusid = record.corpusid
        url = record.text_url
        print('---------------------------------------------------')
        print(corpusid)
        print(url)
        symptom = record.Symptom

        # Find the index of the references keyword and truncate the text
        references_index = text.find(references_keyword)
        if references_index != -1:
            text = text[:references_index]

        # Extract information using NER_2 and convert to DataFrame row
        result_all = NER_2_updates(text, df_BBDD_nonprocessed_example, df_BBDD_example)
        print(result_all)
        result_periodicity = NER_2_periodicity(text, df_BBDD_nonprocessed_example, df_BBDD_example)
        print(result_periodicity)
        result_sociodemo = NER_2_sociodemo(text, df_BBDD_nonprocessed_example, df_BBDD_example)
        print(result_sociodemo)
        print('')

        extracted_info_all = extract_info_from_json(result_all)
        print(extracted_info_all)
        extracted_info_periodicity = extract_info_from_json_periodicity(result_periodicity)
        print(extracted_info_periodicity)
        extracted_info_sociodemo = extract_info_from_json_sociodemo(result_sociodemo)
        print(extracted_info_sociodemo)
        print('')

        combined_info = combine_extracted_info(extracted_info_all, extracted_info_periodicity, extracted_info_sociodemo)
        print(combined_info)
        print('---------------------------------------------------')
        combined_info['corpusid'] = corpusid
        combined_info['text_url'] = url
        df_protocolo_parameters.append(combined_info)

    # Convert list of dictionaries to DataFrame
    df_protocolo_parameters = pd.DataFrame(df_protocolo_parameters)

    return df_protocolo_parameters

### **Actualizaciones Periódicas**

Esta sección está destinada a realizar actualizaciones regulares de la base de datos de protocolos. Con una frecuencia determinada por el criterio del profesional clínico (se recomienda cada 15 días/ 1 mes), se extraerán nuevos protocolos que hayan sido incorporados en Semantic Scholar. Estos protocolos se almacenarán de forma temporal como parte de un historial, y solo serán integrados oficialmente en el sistema una vez cuenten con la aprobación del clínico.

#### **Extracción de nuevos releases y sus documentos relacionados con tDCS**

En esta sección se hace una llamada a la API de Semantic Scholar y **se recuperan todos los artículos relacionados con tDCS** en su base de datos. Posteriormente, se filtra para determinar los artículos nuevos a incorporar en nuestra base de datos.

Además, en esta parte del proceso se incorporan una serie de cambios a la hora de extraer el texto de los artículos relacionados con tDCS. De esta manera, de la lista de artículos con sus respectivos urls de acceso abierto, **se descarga su contenido** (en una carpeta temporal) **y se extrae el texto** de forma automática.

Para los documentos que no se han podido descargar de automáticamente, se hará uso de un **código auxiliar** a ejecutar desde la terminal (local).

*Para más leer la DOCUMENTACIÓN (Sección 2.3.1.2).*

In [ ]:
def initializer(path_historial, path_updates, path_BBDD):
  """
  Initializes the historical, updates and deletes tables, and database. Initializes empty dictionaries to handle deletion and update information.
  """

  historial_1 = pd.read_excel(path_historial)

  updates_table_1 = pd.read_excel(path_updates)

  BBDD_1 = pd.read_excel(path_BBDD)


  # Update dictionary -- restarts with each release (to put in history)
  updates_dict_init = {
      'corpusid': [],
      'DOI': [],
      'Title': [],
      'Evidence anterior': [],
      'Evidence_level anterior': [],
      'Evidence nuevo': [],
      'Evidence_level nuevo': []
  }
  # Dictionary with the information of each new update -- restarts with each new (temporary) update.
  new_update_dict_init = {
      'release': [],
      'corpusid': [],
      'DOI': [],
      'Title': [],
      'Evidence anterior': [],
      'Evidence_level anterior': [],
      'Evidence nuevo': [],
      'Evidence_level nuevo': []
  }

  return historial_1, updates_table_1, BBDD_1, updates_dict_init, new_update_dict_init

In [ ]:
def extract_releases(historial):
  """
  Extracts new releases from Semantic Scholar, and their specific differences (updates and deletes) between last and new release.
  """
  print('Extrayendo las actualizaciones de Semantic Scholar: ')

  # Extract the latest release from the historical
  if historial.empty:
    last_release = '2024-04-23'
  else:
    last_release = historial['Release nuevo'].iloc[-1]
  print('La última actualización fue en la fecha: ', last_release)

  # Get the list of tDCS papers in Semantic Scholar
  from semanticscholar import SemanticScholar
  sch = SemanticScholar()
  results = sch.search_paper('tDCS', open_access_pdf=True, bulk=True)
  all_results = [item for item in results]
  print('Número de artículos de tDCS en Semantic Scholar: ', len(all_results))

  return all_results

In [ ]:
def updates_folder_creation(last_release, dtset_updates_dir_or):
  """
  Creates folders for the corpus of new updates.
  """
  # Create a download folder for the updates of that release (if it does not exist).
  dtset_updates_dir_1 = Path(dtset_updates_dir_or)
  dtset_updates_dir_1 = dtset_updates_dir_1.joinpath(last_release)

  if not dtset_updates_dir_1.exists():
    dtset_updates_dir_1.mkdir(parents=False, exist_ok=True)
    print(f"Carpeta '{dtset_updates_dir_1}' creada.")
  else:
    print(f"La carpeta '{dtset_updates_dir_1}' ya existe. Saltando creación.")

  return dtset_updates_dir_1

In [ ]:
def save_tdcs_updates(all_results, dtset_updates_dir):
  """
  Saves the JSON of new tDCS papers in Semantic Scholar (extracting only its corpusid and url).
  """
  print('Guardando el JSON de actualizaciones en la carpeta updates: ')
  json_data = []
  for item in all_results:
    corpusid = item['corpusId']
    text_url = item['openAccessPdf']['url']
    json_paper = {'corpusid': corpusid, 'text_url': text_url}
    json_data.append(json_paper)

  # Ruta de guardado en Google Drive
  folder_path = dtset_updates_dir
  os.makedirs(folder_path, exist_ok=True)
  file_path = os.path.join(folder_path, "articulos.json")
  # Guardar como archivo JSON (.json)
  with open(file_path, 'w') as f:
      json.dump(json_data, f, ensure_ascii=False, indent=2)
  print("Artículos guardados en formato .json")

In [ ]:
def check_tdcs_updates(dtset_updates_dir, path_urls_record):
  """
  Checks if there are new tDCS papers in Semantic Scholar (not in our BBDD or processed before).
  """

  print('Comprobando el listado de urls: ')
  papers_tdcs = []
  urls_nuevos = []

  # Leemos el json de la carpeta de updates
  file_path = os.path.join(dtset_updates_dir, "articulos.json")
  with open(file_path, 'r') as f:
    articulos = json.load(f)

  # Leemos la lista de urls registrados en actualizaciones pasadas
  urls_list = pd.read_excel(path_urls_record)

  for item in articulos:
    paper_url = item['text_url']
    # Si ya está en nuestra lista de urls -- no es un update (si el mismo paper se actualiza el URL CAMBIA)
    if paper_url in urls_list['url'].values:
      continue
    # Si no está en nuestra lista -- lo añadimos a la lista y registramos como un nuevo paper de tdcs
    else:
      papers_tdcs.append(item)
      urls_nuevos.append(paper_url)

  print('Hay ', len(papers_tdcs), 'papers nuevos de tDCS en Semantic Scholar')
  print('')
  # Guardamos los papers nuevos de tDCS en un json
  file_path = os.path.join(dtset_updates_dir, "articulos_tdcs.json")
  with open(file_path, 'w') as f:
      json.dump(papers_tdcs, f, ensure_ascii=False, indent=2)

  # Actualizamos el registro de urls
  df_urls_nuevos = pd.DataFrame({'url': urls_nuevos})
  urls_list = pd.concat([urls_list, df_urls_nuevos], ignore_index=True)
  urls_list.to_excel(path_urls_record, index=False)

In [ ]:
def extract_text_from_pdf_url(url, save_path="/tmp/temp_paper.pdf"):
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"
        }

        response = requests.get(url, headers=headers, stream=True, timeout=15)

        content_type = response.headers.get("Content-Type", "").lower()
        if response.status_code == 200 and ("application/pdf" in content_type or "application/octet-stream" in content_type or url.endswith(".pdf")):
            with open(save_path, "wb") as f:
                f.write(response.content)

            # Extraer texto del PDF
            try:
                text = ""
                with fitz.open(save_path) as doc:
                    for page in doc:
                        text += page.get_text()
                return text.strip()
            except Exception as e:
                return ""
        else:
            return ""

    except Exception as e:
        return ""


def extract_text_updates(dir_papers):

    try:
        dir_papers_tdcs = dir_papers.joinpath("articulos_tdcs.json")
        with open(dir_papers_tdcs, 'r', encoding='utf-8') as f:
            data = json.load(f)

        for paper in data:
            url = paper.get("text_url")
            if url:
                paper["text"] = extract_text_from_pdf_url(url)
            else:
                paper["text"] = ""

        # Crear Spark DataFrame de los textos de tdcs
        try:
          spark_df = spark.createDataFrame(data)
        except:
          schema = StructType([
              StructField("corpusid", StringType(), True),
              StructField("text", StringType(), True),
              StructField("text_url", StringType(), True)
          ])
          spark_df = spark.createDataFrame([], schema)

    except Exception as e:
        print(f'Unexpected error: {e}')
        return None

    return spark_df

In [ ]:
# FUNCIÓN PRINCIPAL
def extract_release_tdcs(path_historial, path_updates, path_BBDD, dtset_updates_dir_or, path_urls_record, path_rpa_papers):
  """
  Extracts new releases from Semantic Scholar, checks for new updates, and extracts their texts.
  """

  # Initializing
  historial, updates_table, BBDD, updates_dict_init, new_update_dict_init = initializer(path_historial, path_updates, path_BBDD)
  # New releases extraction
  results = extract_releases(historial)

  if historial.empty:
    id_historial_last = 'HA0'
    id_historial_last = id_historial_last[2:]
    id_historial = int(id_historial_last)
  else:
    id_historial_last = historial.iloc[-1]['ID historial']
    id_historial_last = id_historial_last[2:]
    id_historial = int(id_historial_last)

  # Processing
  prev_release = historial['Release nuevo'].iloc[-1]
  last_release = datetime.today().strftime('%Y-%m-%d')
  last_release = str(last_release)
  print("Buscando las actualizaciones entre el release ", prev_release , " y el release ",  last_release)
  print('')

  # Folder creation
  dtset_updates_dir = updates_folder_creation(last_release, dtset_updates_dir_or)
  updates_total = len(results)
  print('Número de corpus UPDATES: ', updates_total)
  print('')
  #Saving total updates in updates folder
  save_tdcs_updates(results, dtset_updates_dir)

  # Text extraction
  check_tdcs_updates(dtset_updates_dir, path_urls_record) # Comprobamos el listado de urls y guardamos los nuevos en un json
  df_tdcs_autom = extract_text_updates(dtset_updates_dir) # Descarga y extracción de textos

  if df_tdcs_autom.count() != 0:

    # Filtramos el dataframe de spark por los papers con text vacío (no se han podido descargar)
    df_tdcs_vacios = df_tdcs_autom.filter((col('text').isNull()) | (trim(col('text')) == ''))
    print('No se ha podido extraer el texto de ', df_tdcs_vacios.count(), 'papers')
    df_tdcs_vacios = df_tdcs_vacios.toPandas()
    # Guardamos los nuevos papers de texto vacío en el registro para descargar con RPA
    df_tdcs_vacios_record = pd.read_excel(path_rpa_papers)
    df_tdcs_vacios_all = pd.concat([df_tdcs_vacios_record, df_tdcs_vacios], ignore_index=True)
    if 'Unnamed: 0' in df_tdcs_vacios_all.columns:
      df_tdcs_vacios_all = df_tdcs_vacios_all.drop(columns=['Unnamed: 0'])
    df_tdcs_vacios_all.to_excel(path_rpa_papers)

    # Guardamos los nuevos papers con texto extraido automaticamente (no vacío)
    df_tdcs_autom_cleaned = df_tdcs_autom.filter((col('text').isNotNull()) & (trim(col('text')) != ''))
    path_tdcs_filtered = dtset_updates_dir.joinpath('df_tdcs_autom')
    if df_tdcs_autom_cleaned.count() != 0:
      df_tdcs_autom_cleaned.write.parquet(path_tdcs_filtered.as_posix())
    else:
      print('-------------------------------------------------------------------------------------------------------------------------------------')
      print('No hay nuevos papers con texto extraido automaticamente. Revisar archivo df_tdcs_vacios.xlsx para ver actualizaciones manuales (RPA).')
      print('-------------------------------------------------------------------------------------------------------------------------------------')

  else:
    print('------------------------------')
    print('No hay nuevas actualizaciones.')
    print('------------------------------')

#### **Descarga Auxiliar con RPA**

Para los documentos que no se han podido descargar de forma automática, se hará uso de un **código auxiliar** a ejecutar desde la terminal (local).

Al finalizar la ejecución, el dataframe creado se deberá guardar en la misma carpeta que el dataframe anterior.

*Para más leer la DOCUMENTACIÓN (Sección 2.3.1.2).*

#### **Procesamiento de Actualizaciones**

Tras realizar ambas descargas se procede a hacer el procesamiento de las actualizaciones, y registrar los cambios en los archivos Historial y Tabla de actualizaciones para su posterior revisado por el clínico.

In [ ]:
def filter_kw(dataframe, text_column, keyword_list):  ## Filter function intended just for the 'tDCS' word
    def count_kwds(text):
        if text is None:
            return 0
        else:
            return sum(text.lower().count(k) for k in keyword_list)

    count_kwds_udf = udf(count_kwds, IntegerType())
    dataset = dataframe.withColumn("Kwd_count", count_kwds_udf(dataframe[text_column]))
    dataset = dataset.filter(dataset.Kwd_count > 25).cache()
    return dataset

In [ ]:
from pyspark.sql.functions import lit

def filter_symptoms_updates(dataframe, text_column, symptom_keywords, end_index, threshold):

    filtered_dataframe = filter_kwd(dataframe, text_column, symptom_keywords, end_index, threshold)
    updated_rows = []

    #symptom_caused = {}
    symptom_list = []
    counter_list = []
    for row in filtered_dataframe.collect():
      if row[text_column] is not None:
          #print('-------------')
          #print('NEW ROW')
          corpusid = row['corpusid']
          text_url = row['text_url']
          text = row[text_column][:end_index]
          symptom_caused = {}
          max_symptom = 0
          max_counter = 0
          for keyword in symptom_keywords:
              counter = text.lower().count(keyword.lower())
              if counter >= (threshold):
                  if counter > max_counter:
                      max_counter = counter
                      max_symptom = keyword
          updated_rows.append((corpusid, text_url, text, max_symptom, max_counter))
      else:
         continue

    # Definir el esquema del DataFrame
    schema = StructType([
        StructField("corpusid", StringType(), True),
        StructField("text_url", StringType(), True),
        StructField("text", StringType(), True),
        StructField("Symptom", StringType(), True),
        StructField("counter", IntegerType(), True)
    ])

    # Crear el DataFrame con el esquema definido
    df_updated = spark.createDataFrame(updated_rows, schema)
    df_updated = df_updated.filter(col("counter") > 1)

    return df_updated

In [ ]:
def processing_updates(dtset_updates_dir, BBDD, updates_dict_init, new_update_dict_init, last_release, updates_table, path_updates, keyword_list_symptoms, path_urls_record, df_tdcs_joined):
  """
  Processes the corpus of new updates, and saves the information in the updates table.
  """

  # First filter: tDCS related
  keywords = ['tdcs']
  df_filtered_tdcs_keyword = filter_kw(df_tdcs_joined, 'text', keywords) # Contabilizamos y filtramos por 'tdcs'

  updates_tdcs_1 = df_filtered_tdcs_keyword.count()
  print('Número de protocolos de tDCS a actualizar:', updates_tdcs_1)
  print('')
  path_tdcs_filtered = dtset_updates_dir.joinpath('df_filtered_tdcs_joined')
  df_filtered_tdcs_keyword.write.parquet(path_tdcs_filtered.as_posix())

  # Second filter: specific symptoms
  keyword_list_1= keyword_list_symptoms
  df_text_symptoms = filter_symptoms_updates(df_filtered_tdcs_keyword, 'text', keyword_list_1,end_index= 4000, threshold=6)
  updates_symptoms_1 = int(df_text_symptoms.count())
  print('Número de protocolos con síntomas específicos a actualizar:', updates_symptoms_1)

  if updates_symptoms_1 != 0:

    # GPT NER PROMPT
    df_protocolo_parameters = parameter_extraction(df_text_symptoms)

    # Parameters standardization
    df_protocolo_parameters_copy = df_protocolo_parameters.copy()
    df_standarized = standarization(df_protocolo_parameters_copy)

    # Save the DataFrame as a Parquet file
    output_parquet_path = dtset_updates_dir.joinpath(f"df_standarized_{str(last_release)}")
    df_standarized.to_csv(output_parquet_path)
    print(f"DataFrame GPT NER estandarizado saved as Parquet file at: {output_parquet_path}")

    # Updates table updating
    updates_dict_1 = {
        'corpusid': [],
        'DOI': [],
        'Title': [],
        'Evidence anterior': [],
        'Evidence_level anterior': [],
        'Evidence nuevo': [],
        'Evidence_level nuevo': []
      }

    for index, row in df_standarized.iterrows():

        # New update dictionary
        new_update_dict = {
            'release': [],
            'corpusid': [],
            'DOI': [],
            'Title': [],
            'Evidence anterior': [],
            'Evidence_level anterior': [],
            'Evidence nuevo': [],
            'Evidence_level nuevo': []
        }

        release = last_release
        new_update_dict['release'].append(release)

        corpusid = str(row['corpusid'])
        print(corpusid)
        new_update_dict['corpusid'].append(corpusid)
        updates_dict_1['corpusid'].append(corpusid)

        Title = row['Title']
        print(Title)
        new_update_dict['Title'].append(Title)
        updates_dict_1['Title'].append(Title)

        DOI = row['DOI']
        new_update_dict['DOI'].append(DOI)
        updates_dict_1['DOI'].append(DOI)

        # If the corpusid AND TITLE is in our DB -- it is recorded as evidence update
        BBDD['corpusid'] = BBDD['corpusid'].astype(str)
        BBDD['Title'] = BBDD['Title'].astype(str).str.lower()
        Title = str(Title).lower()

        if corpusid in BBDD['corpusid'].values and Title in BBDD['Title'].values:
            print('Está en la base de datos')

            # Extract the evidences
            row_evidencia_anterior = BBDD.loc[BBDD['corpusid'] == corpusid]
            for value in row_evidencia_anterior['Evidence'].values:
              evidencia_anterior = value
              print('La evidencia anterior es', evidencia_anterior)
            for value in row_evidencia_anterior['Evidence_level'].values:
              evidencia_anterior_level = value
              print('El level de la evidencia anterior es', evidencia_anterior_level)

            evidencia_nueva = row['Evidence']
            print('La evidencia nueva es', evidencia_nueva)
            evidencia_nueva_level = row['Evidence_level']
            print('El level de la evidencia nueva es', evidencia_nueva_level)

            # Add the NEW evidences to the information dictionary of the new UPDATE
            new_update_dict['Evidence nuevo'].append(evidencia_nueva)
            new_update_dict['Evidence_level nuevo'].append(evidencia_nueva_level)
            # Add the PREVIOUS evidence
            new_update_dict['Evidence anterior'].append(evidencia_anterior)
            new_update_dict['Evidence_level anterior'].append(evidencia_anterior_level)
            # Do the same with the dictionary of all UPDATES of this new release.
            updates_dict_1['Evidence nuevo'].append(evidencia_nueva)
            updates_dict_1['Evidence_level nuevo'].append(evidencia_nueva_level)
            updates_dict_1['Evidence anterior'].append(evidencia_anterior)
            updates_dict_1['Evidence_level anterior'].append(evidencia_anterior_level)

            print("Protocolo para actualizar: Evidencia anterior: ", evidencia_anterior, evidencia_anterior_level, ". Evidencia nueva: ", evidencia_nueva, evidencia_nueva_level)
            print('')
            print(new_update_dict)

        # If the corpusid is not -- registered as new protocol
        else:
            print('No está en la base de datos')
            evidencia_anterior = '-'
            evidencia_anterior_level = '-'
            evidencia_nueva = row['Evidence']
            evidencia_nueva_level = row['Evidence_level']

            # Add the NEW evidences to the information dictionary of the new UPDATE
            new_update_dict['Evidence nuevo'].append(evidencia_nueva)
            new_update_dict['Evidence_level nuevo'].append(evidencia_nueva_level)
            # Add the PREVIOUS evidence
            new_update_dict['Evidence anterior'].append(evidencia_anterior)
            new_update_dict['Evidence_level anterior'].append(evidencia_anterior_level)
            # Do the same with the dictionary of all UPDATES of this new release.
            updates_dict_1['Evidence nuevo'].append(evidencia_nueva)
            updates_dict_1['Evidence_level nuevo'].append(evidencia_nueva_level)
            updates_dict_1['Evidence anterior'].append(evidencia_anterior)
            updates_dict_1['Evidence_level anterior'].append(evidencia_anterior_level)

            print("Nuevo protocolo a añadir. Con evidencia: ", evidencia_nueva, evidencia_nueva_level)
            print('')
            print(new_update_dict)

        # Include the new update in the update table.
        nuevo_update = pd.DataFrame(new_update_dict)
        updates_table = pd.concat([updates_table, nuevo_update], ignore_index= True)

  else:
    print('No hay actualizaciones con los síntomas específicos. Se procede a evaluar las eliminaciones.')
    updates_dict_1 = {
        'corpusid': [],
        'DOI': [],
        'Title': [],
        'Evidence anterior': [],
        'Evidence_level anterior': [],
        'Evidence nuevo': [],
        'Evidence_level nuevo': []
    }

  # Save Updates Table in Excel file.
  updates_table.to_excel(path_updates, index=False)

  return updates_tdcs_1, updates_symptoms_1, updates_dict_1, path_updates


In [ ]:
def historial_periodic_updates(path_historial, path_updates, path_BBDD, dtset_updates_dir_or, keyword_list_symptoms, path_urls_record, path_rpa_papers):

  """
  Searches new protocols included or excluded from Semantic Scholar. Stores such differences as a historical, until the clinician gives their approval.
  """

  # Initializing
  historial, updates_table, BBDD, updates_dict_init, new_update_dict_init = initializer(path_historial, path_updates, path_BBDD)

  if historial.empty:
    id_historial_last = 'HA0'
    id_historial_last = id_historial_last[2:]
    id_historial = int(id_historial_last)
  else:
    id_historial_last = historial.iloc[-1]['ID historial']
    id_historial_last = id_historial_last[2:]
    id_historial = int(id_historial_last)

  # Processing
  prev_release = historial['Release nuevo'].iloc[-1]
  last_release = datetime.today().strftime('%Y-%m-%d')
  last_release = str(last_release)
  print("Procesando las actualizaciones entre el release ", prev_release , " y el release ",  last_release)
  print('')

  dtset_updates_dir = Path(dtset_updates_dir_or).joinpath(last_release)
  file_path = os.path.join(dtset_updates_dir, "articulos.json")
  with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)
  updates_total = len(data)
  print('Número de corpus UPDATES: ', updates_total)
  print('')

  id_historial = id_historial + 1

  # UPDATES

  # Updates processing
  historial, updates_table, BBDD, updates_dict_init, new_update_dict_init = initializer(path_historial, path_updates, path_BBDD)

  path_df_tdcs_autom = dtset_updates_dir.joinpath('df_tdcs_autom')
  path_df_tdcs_RPA = dtset_updates_dir.joinpath('df_tdcs_RPA')

  if os.path.exists(path_df_tdcs_autom):
    df_tdcs_autom = spark.read.parquet(path_df_tdcs_autom.as_posix())
    # Si tenemos papers automaticos y de RPA -- unimos, procesamos y limpiamos el registro de papers RPA
    if os.path.exists(path_df_tdcs_RPA):
      print('Procesando actualizaciones (automáticas y manuales).')
      df_tdcs_RPA = spark.read.parquet(path_df_tdcs_RPA.as_posix())
      df_tdcs_joined = df_tdcs_autom.union(df_tdcs_RPA)
      updates_tdcs, updates_symptoms, updates_dict, path_updates = processing_updates(dtset_updates_dir, BBDD, updates_dict_init, new_update_dict_init, last_release, updates_table, path_updates, keyword_list_symptoms, path_urls_record, df_tdcs_joined)
      df_tdcs_vacios = pd.DataFrame(columns = ['corpusid', 'text', 'text_url'])
      df_tdcs_vacios.to_excel(path_rpa_papers)
    # Si solo hay papers automaticos -- procesamos esos únicamente
    else:
      print('Procesando actualizaciones (solo hay automáticas).')
      updates_tdcs, updates_symptoms, updates_dict, path_updates = processing_updates(dtset_updates_dir, BBDD, updates_dict_init, new_update_dict_init, last_release, updates_table, path_updates, keyword_list_symptoms, path_urls_record, df_tdcs_autom)
  else:
    # Si solo hay papers de RPA -- procesamos esos únicamente y limpiamos el registro de papers RPA
    if os.path.exists(path_df_tdcs_RPA):
      print('Procesando actualizaciones (solo hay manuales).')
      df_tdcs_RPA = spark.read.parquet(path_df_tdcs_RPA.as_posix())
      updates_tdcs, updates_symptoms, updates_dict, path_updates = processing_updates(dtset_updates_dir, BBDD, updates_dict_init, new_update_dict_init, last_release, updates_table, path_updates, keyword_list_symptoms, path_urls_record, df_tdcs_RPA)
      df_tdcs_vacios = pd.DataFrame(columns = ['corpusid', 'text', 'text_url'])
      df_tdcs_vacios.to_excel(path_rpa_papers)
    # Si no hay papers nuevos (no automáticos ni RPA) -- No se procesan las actualizaciones
    else:
      print('----------------------------------------------------------------------------------------------------------------')
      print('No hay nuevas actualizaciones. Revisar archivo df_tdcs_vacios.xlsx para comprobar actualizaciones no realizadas.')
      print('----------------------------------------------------------------------------------------------------------------')
      updates_tdcs = 0
      updates_symptoms = 0
      updates_dict = {
          'corpusid': [],
          'DOI': [],
          'Title': [],
          'Evidence anterior': [],
          'Evidence_level anterior': [],
          'Evidence nuevo': [],
          'Evidence_level nuevo': []
      }

  # Historical Updating
  nuevo_registro ={
      'ID historial': ["HA" + str(id_historial)],
      'Release anterior': [prev_release],
      'Release nuevo': [last_release],
      '# UPDATES total': [updates_total],
      '# UPDATES tDCS': [updates_tdcs],
      '# UPDATES con síntomas': [updates_symptoms],
      'UPDATES info': [updates_dict]
      }

  nuevo_registro = pd.DataFrame(nuevo_registro)
  historial = pd.concat([historial, nuevo_registro], ignore_index= True)

  # Save Historical in Excel file
  historial.to_excel(path_historial, index=False)

#### **Ejecución de Actalizaciones Periódicas**

Aproximadamente cada mes, este proceso de actualización se deberá ejecutar en el siguiente orden:


1.   Recuperación y descarga de actualizaciones automáticas
2.   Descarga auxiliar con RPA (local)
3.   Procesamiento de actualizaciones



In [ ]:
# RECUPERACIÓN Y DESCARGA DE ACTUALIZACIONES AUTOMÁTICA
extract_release_tdcs(path_historial, path_updates, path_BBDD, dtset_updates_dir_or, path_urls_record, path_rpa_papers)

Extrayendo las actualizaciones de Semantic Scholar: 
La última actualización fue en la fecha:  2025-04-27
Número de artículos de tDCS en Semantic Scholar:  5887
Buscando las actualizaciones entre el release  2025-04-27  y el release  2025-05-12

La carpeta '/content/drive/MyDrive/ALTERNATIVAS_SEMANTIC_SCHOLAR/updates/2025-05-12' ya existe. Saltando creación.
Número de corpus UPDATES:  5887

Guardando el JSON de actualizaciones en la carpeta updates: 
Artículos guardados en formato .json
Comprobando el listado de urls: 
Hay  2 papers nuevos de tDCS en Semantic Scholar

No se ha podido extraer el texto de  2 papers
No hay nuevos papers con texto extraido automaticamente. Revisar archivo df_tdcs_vacios.xlsx para ver actualizaciones manuales (RPA).


In [ ]:
# PROCESAMIENTO DE ACTUALIZACIONES
historial_periodic_updates(path_historial, path_updates, path_BBDD, dtset_updates_dir_or, keyword_list_symptoms, path_urls_record, path_rpa_papers)

Procesando las actualizaciones entre el release  2025-04-27  y el release  2025-05-12

Número de corpus UPDATES:  5887

Procesando actualizaciones (solo hay manuales).
Número de protocolos de tDCS a actualizar: 1

Número de protocolos con síntomas específicos a actualizar: 0
No hay actualizaciones con los síntomas específicos. Se procede a evaluar las eliminaciones.
